In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for reference (UTC+8) as required (do not convert timestamps per instructions)
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files into memory (keep full tables in variables for later use)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# --- metric.csv aggregates ---
metric_min_max = (metric_df['timestamp'].min(), metric_df['timestamp'].max())

metric_cmdb_count = metric_df['cmdb_id'].nunique()
metric_cmdb_unique = pd.DataFrame({'cmdb_id': metric_df['cmdb_id'].unique()[:50]})

metric_kpi_count = metric_df['kpi_name'].nunique()
metric_kpi_unique = pd.DataFrame({'kpi_name': metric_df['kpi_name'].unique()[:50]})

# per-(cmdb_id,kpi_name) aggregates: count, P50, P95, max(value)
metric_group = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count='count',
         p50=lambda x: x.quantile(0.5),
         p95=lambda x: x.quantile(0.95),
         max_value='max')
    .sort_values('count', ascending=False)
    .head(50)
)

# --- trace.csv aggregates ---
trace_min_max = (trace_df['timestamp'].min(), trace_df['timestamp'].max())

trace_cmdb_count = trace_df['cmdb_id'].nunique()
trace_cmdb_unique = pd.DataFrame({'cmdb_id': trace_df['cmdb_id'].unique()[:50]})

trace_name_count = trace_df['trace_name'].nunique()
trace_name_unique = pd.DataFrame({'trace_name': trace_df['trace_name'].unique()[:50]})

trace_group = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count='count',
         p50=lambda x: x.quantile(0.5),
         p95=lambda x: x.quantile(0.95),
         max_value='max')
    .sort_values('count', ascending=False)
    .head(50)
)

# --- log.csv aggregates ---
log_min_max = (log_df['timestamp'].min(), log_df['timestamp'].max())

log_cmdb_count = log_df['cmdb_id'].nunique()
log_cmdb_unique = pd.DataFrame({'cmdb_id': log_df['cmdb_id'].unique()[:50]})

log_name_count = log_df['log_name'].nunique()
log_name_unique = pd.DataFrame({'log_name': log_df['log_name'].unique()[:50]})

log_group = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(count='count',
         p50=lambda x: x.quantile(0.5),
         p95=lambda x: x.quantile(0.95),
         max_value='max')
    .sort_values('count', ascending=False)
    .head(50)
)

# --- error_logs.csv aggregates ---
error_min_max = (error_df['timestamp'].min(), error_df['timestamp'].max())

error_cmdb_count = error_df['cmdb_id'].nunique()
error_cmdb_unique = pd.DataFrame({'cmdb_id': error_df['cmdb_id'].unique()[:50]})

error_samples = error_df[['timestamp', 'cmdb_id', 'message']].head(20)

# Compact final display (variables shown below)
metric_min_max, metric_cmdb_count, metric_cmdb_unique, metric_kpi_count, metric_kpi_unique, metric_group, \
trace_min_max, trace_cmdb_count, trace_cmdb_unique, trace_name_count, trace_name_unique, trace_group, \
log_min_max, log_cmdb_count, log_cmdb_unique, log_name_count, log_name_unique, log_group, \
error_min_max, error_cmdb_count, error_cmdb_unique, error_samples

```
Out[1]:
```
```python
# Compose a concise plain-English summary string using existing kernel variables
metric_min, metric_max = int(metric_min_max[0]), int(metric_min_max[1])
trace_min, trace_max = int(trace_min_max[0]), int(trace_min_max[1])
log_min, log_max = int(log_min_max[0]), int(log_min_max[1])
error_min, error_max = int(error_min_max[0]), int(error_min_max[1])

def sample_list(arr, n=6):
    return ', '.join(map(str, list(arr[:n])))

metric_examples = sample_list(metric_df['cmdb_id'].unique(), 6)
metric_kpi_examples = sample_list(metric_df['kpi_name'].unique(), 3)

trace_examples = sample_list(trace_df['cmdb_id'].unique(), 6)
trace_name_examples = sample_list(trace_df['trace_name'].unique(), 3)

log_examples = sample_list(log_df['cmdb_id'].unique(), 6)
log_name_examples = sample_list(log_df['log_name'].unique(), 3)

error_examples = sample_list(error_df['cmdb_id'].unique(), 6)

top_metric_count = int(metric_group['count'].iloc[0]) if not metric_group.empty else 0
top_trace_count = int(trace_group['count'].iloc[0]) if not trace_group.empty else 0
top_log_count = int(log_group['count'].iloc[0]) if not log_group.empty else 0

summary = (
    "Summary of loaded telemetry (timestamps are integer seconds, UTC+8 context):\n\n"
    f"metric.csv: timestamp range {metric_min} - {metric_max}; "
    f"{metric_cmdb_count} unique cmdb_id (examples: {metric_examples}); "
    f"{metric_kpi_count} unique kpi_name (examples: {metric_kpi_examples}).\n"
    f"Per-(cmdb_id,kpi_name) aggregates computed (count, P50, P95, max); returned top 50 rows by count (top count = {top_metric_count}).\n\n"
    f"trace.csv: timestamp range {trace_min} - {trace_max}; "
    f"{trace_cmdb_count} unique cmdb_id (examples: {trace_examples}); "
    f"{trace_name_count} unique trace_name (examples: {trace_name_examples}).\n"
    f"Per-(cmdb_id,trace_name) aggregates computed (count, P50, P95, max); returned top 50 rows by count (top count = {top_trace_count}).\n\n"
    f"log.csv: timestamp range {log_min} - {log_max}; "
    f"{log_cmdb_count} unique cmdb_id (examples: {log_examples}); "
    f"{log_name_count} unique log_name (examples: {log_name_examples}).\n"
    f"Per-(cmdb_id,log_name) aggregates computed (count, P50, P95, max); returned top 50 rows by count (top count = {top_log_count}).\n\n"
    f"error_logs.csv: timestamp range {error_min} - {error_max}; "
    f"{error_cmdb_count} unique cmdb_id (examples: {error_examples}). "
    "Up to 20 sample error rows were returned (timestamp, cmdb_id, message).\n\n"
    "Notes: Full tables remain in kernel variables (metric_df, trace_df, log_df, error_df). "
    "Displayed aggregates were kept compact and limited as requested."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((np.int64(1616589000), np.int64(1616590800)), 26,      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4    Mysql01
..       ...
21  Tomcat03
22  Tomcat04
23  apache01
24  apache02
25  dockerA2

[26 rows x 1 columns], 356,                                              kpi_name
0            JVM-Memory_7778_JVM_Memory_HeapMemoryMax
1          JVM-Memory_7778_JVM_Memory_HeapMemoryUsage
2           JVM-Memory_7778_JVM_Memory_HeapMemoryUsed
3         JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed
4                     JVM-Runtime_7778_JVM_JVM_Uptime
..                                                ...
45  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvg...
46  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...
47    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps
48    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead
49    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps

[50 rows x 1 columns],        cmdb_id                                           kpi_name  count           p50           p95     max_value
1568  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00
1569  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00
1570  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00
1571  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00
1572  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00
...        ...                                                ...    ...           ...           ...           ...
87        IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09
1435  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  1.093000e+03  1.093000e+03  1.093000e+03
1436  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  4.016560e+05  4.016560e+05  4.016560e+05
1437  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  2.038697e+09  2.039467e+09  2.039580e+09
1438  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  3.175364e+06  3.178084e+06  3.178414e+06

[50 rows x 6 columns], (np.int64(1616589000), np.int64(1616590800)), 12,      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4   Tomcat01
5   Tomcat02
6   Tomcat03
7   Tomcat04
8   dockerA1
9   dockerA2
10  dockerB1
11  dockerB2, 72,                          trace_name
0          trace.self.duration_mean
1           trace.self.duration_p95
2              trace.self.row_count
3   trace.to_Tomcat01.duration_mean
4    trace.to_Tomcat01.duration_p95
..                              ...
45  trace.to_dockerB1.duration_mean
46   trace.to_dockerB1.duration_p95
47      trace.to_dockerB1.row_count
48  trace.to_dockerB2.duration_mean
49   trace.to_dockerB2.duration_p95

[50 rows x 1 columns],     cmdb_id                        trace_name  count         p50         p95   max_value
0      IG01          trace.self.duration_mean     31    0.254777    3.975726    4.314705
1      IG01           trace.self.duration_p95     31    1.135450   15.579500   15.584100
2      IG01              trace.self.row_count     31  752.000000  942.000000  972.000000
3      IG01   trace.to_Tomcat01.duration_mean     31    0.223463    0.602815    1.044000
4      IG01    trace.to_Tomcat01.duration_p95     31    1.078450    1.929100    5.327200
..      ...                               ...    ...         ...         ...         ...
101    MG02       trace.to_dockerA2.row_count     31   68.000000   95.000000  100.000000
102    MG02   trace.to_dockerB1.duration_mean     31    0.203944    0.428772    0.574615
80     MG02     trace.from_Tomcat04.row_count     31   96.000000  124.500000  130.000000
91     MG02  trace.from_dockerB2.duration_p95     31    0.019400    2.895400    5.109600
89     MG02     trace.from_dockerB1.row_count     31  126.000000  216.500000  242.000000

[50 rows x 6 columns], (np.int64(1616589000), np.int64(1616590800)), 6,     cmdb_id
0  Tomcat01
1  Tomcat02
2  Tomcat03
3  Tomcat04
4  apache01
5  apache02, 3,            log_name
0   log.error_count
1     log.row_count
2  log.gc_pause_max,      cmdb_id          log_name  count     p50      p95  max_value
0   Tomcat01   log.error_count     31    0.00    0.000       0.00
2   Tomcat01     log.row_count     31  197.00  256.000     265.00
3   Tomcat02   log.error_count     31    0.00    0.000       0.00
5   Tomcat02     log.row_count     31  188.00  231.000     241.00
12  apache01   log.error_count     31    0.00    0.000      19.00
13  apache01     log.row_count     31  381.00  466.000     478.00
11  Tomcat04     log.row_count     31  195.00  234.500     269.00
9   Tomcat04   log.error_count     31    0.00    0.000       0.00
14  apache02   log.error_count     31    0.00    1.500      28.00
15  apache02     log.row_count     31  388.00  472.500     501.00
8   Tomcat03     log.row_count     27  192.00  237.500     257.00
6   Tomcat03   log.error_count     27    0.00    0.000       0.00
10  Tomcat04  log.gc_pause_max     11    0.07    0.100       0.12
1   Tomcat01  log.gc_pause_max      9    0.07    0.086       0.09
4   Tomcat02  log.gc_pause_max      9    0.08    0.080       0.08
7   Tomcat03  log.gc_pause_max      7    0.05    0.094       0.10, (np.int64(1616589452), np.int64(1616589834)), 2,     cmdb_id
0  apache01
1  apache02,      timestamp   cmdb_id                                            message
0   1616589452  apache01  IPAddress "POST /UOCP/person/ServiceTest1.json...
1   1616589452  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
2   1616589452  apache01  IPAddress "POST /UOCP/person/ServiceTest1.json...
3   1616589454  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
4   1616589455  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
5   1616589455  apache01  IPAddress "POST /UOCP/person/ServiceTest1.json...
6   1616589455  apache02  IPAddress "POST /UOCP/person/ServiceTest3.json...
7   1616589455  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
8   1616589455  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
9   1616589456  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...
10  1616589456  apache01  IPAddress "POST /UOCP/person/ServiceTest1.json...
11  1616589456  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
12  1616589456  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...
13  1616589457  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
14  1616589457  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
15  1616589458  apache01  IPAddress "POST /UOCP/person/ServiceTest1.json...
16  1616589458  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...
17  1616589458  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
18  1616589459  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
19  1616589459  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing kernel variables: metric_df, trace_df, log_df, error_df
tz = pytz.timezone('Asia/Shanghai')

# 1) define incident window in UTC+8 and convert to epoch seconds (int)
start_dt = tz.localize(datetime(2021, 3, 24, 20, 30, 0))
end_dt   = tz.localize(datetime(2021, 3, 24, 21, 0, 0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

# 1) Filter tables to the incident window and keep them in kernel variables
metric_window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()
trace_window_df  = trace_df[(trace_df['timestamp']  >= start_ts) & (trace_df['timestamp']  <= end_ts)].copy()
log_window_df    = log_df[(log_df['timestamp']    >= start_ts) & (log_df['timestamp']    <= end_ts)].copy()

# 2) METRICS: compute global thresholds (P95 and P5) per (cmdb_id,kpi_name) BEFORE filtering
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95),
         global_p5 =lambda x: x.quantile(0.05))
)

# Merge thresholds with window series
metric_merged = metric_window_df.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')

# High anomalies: value >= global_p95
metric_high = metric_merged[metric_merged['value'] >= metric_merged['global_p95']].copy()
if not metric_high.empty:
    metric_high_agg = (
        metric_high
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(anomaly_count=('value','count'),
             earliest_ts=('timestamp','min'),
             window_max_value=('value','max'),
             global_threshold=('global_p95','first'))
    )
    metric_high_agg['anomaly_type'] = 'high'
    metric_high_agg['severity'] = metric_high_agg.apply(
        lambda r: np.inf if r['global_threshold'] == 0 else r['window_max_value'] / r['global_threshold'],
        axis=1
    )
else:
    metric_high_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_ts','window_max_value','global_threshold','anomaly_type','severity'])

# Low anomalies: value <= global_p5
metric_low = metric_merged[metric_merged['value'] <= metric_merged['global_p5']].copy()
if not metric_low.empty:
    metric_low_agg = (
        metric_low
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(anomaly_count=('value','count'),
             earliest_ts=('timestamp','min'),
             window_min_value=('value','min'),
             global_threshold=('global_p5','first'))
    )
    metric_low_agg['anomaly_type'] = 'low'
    metric_low_agg['severity'] = metric_low_agg.apply(
        lambda r: np.inf if r['window_min_value'] == 0 else r['global_threshold'] / r['window_min_value'],
        axis=1
    )
    # align column names with high for concatenation
    metric_low_agg = metric_low_agg.rename(columns={'window_min_value':'window_max_value'})
else:
    metric_low_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_ts','window_max_value','global_threshold','anomaly_type','severity'])

# concat and select top 20 by anomaly_count desc then severity desc
metric_anomalies = pd.concat([metric_high_agg, metric_low_agg], ignore_index=True, sort=False)
metric_anomalies_top20 = metric_anomalies.sort_values(['anomaly_count','severity'], ascending=[False,False]).head(20)

# select compact columns and ensure order
metric_anomalies_top20 = metric_anomalies_top20[[
    'cmdb_id','kpi_name','anomaly_type','anomaly_count','earliest_ts','window_max_value','global_threshold','severity'
]]

# 3) TRACES: compute global thresholds per (cmdb_id,trace_name) BEFORE filtering
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95),
         global_p5 =lambda x: x.quantile(0.05))
)

trace_merged = trace_window_df.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# classify trace KPIs: durations -> high if >= P95; row_count -> low if <= P5
is_duration = trace_merged['trace_name'].str.contains('duration', case=False, na=False)
is_rowcount = trace_merged['trace_name'].str.contains('row_count', case=False, na=False)

# duration anomalies (high)
trace_high = trace_merged[is_duration & (trace_merged['value'] >= trace_merged['global_p95'])].copy()
if not trace_high.empty:
    trace_high_agg = (
        trace_high
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(anomaly_count=('value','count'),
             earliest_ts=('timestamp','min'),
             window_max_value=('value','max'),
             global_threshold=('global_p95','first'))
    )
    trace_high_agg['anomaly_type'] = 'high'
    trace_high_agg['severity'] = trace_high_agg.apply(
        lambda r: np.inf if r['global_threshold'] == 0 else r['window_max_value'] / r['global_threshold'],
        axis=1
    )
else:
    trace_high_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_ts','window_max_value','global_threshold','anomaly_type','severity'])

# row_count anomalies (low)
trace_low = trace_merged[is_rowcount & (trace_merged['value'] <= trace_merged['global_p5'])].copy()
if not trace_low.empty:
    trace_low_agg = (
        trace_low
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(anomaly_count=('value','count'),
             earliest_ts=('timestamp','min'),
             window_min_value=('value','min'),
             global_threshold=('global_p5','first'))
    )
    trace_low_agg['anomaly_type'] = 'low'
    trace_low_agg['severity'] = trace_low_agg.apply(
        lambda r: np.inf if r['window_min_value'] == 0 else r['global_threshold'] / r['window_min_value'],
        axis=1
    )
    trace_low_agg = trace_low_agg.rename(columns={'window_min_value':'window_max_value'})
else:
    trace_low_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_ts','window_max_value','global_threshold','anomaly_type','severity'])

trace_anomalies = pd.concat([trace_high_agg, trace_low_agg], ignore_index=True, sort=False)
trace_anomalies_top20 = trace_anomalies.sort_values(['anomaly_count','severity'], ascending=[False,False]).head(20)

# select compact columns
trace_anomalies_top20 = trace_anomalies_top20[[
    'cmdb_id','trace_name','anomaly_type','anomaly_count','earliest_ts','window_max_value','global_threshold','severity'
]]

# 4) LOGS: compute global thresholds per (cmdb_id,log_name) BEFORE filtering
log_thresholds = (
    log_df
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95),
         global_p5 =lambda x: x.quantile(0.05))
)

log_merged = log_window_df.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')

# classify and detect
is_error_count = log_merged['log_name'] == 'log.error_count'
is_log_rowcount = log_merged['log_name'] == 'log.row_count'

# errors: high anomalies
log_high = log_merged[is_error_count & (log_merged['value'] >= log_merged['global_p95'])].copy()
if not log_high.empty:
    log_high_agg = (
        log_high
        .groupby(['cmdb_id','log_name'], as_index=False)
        .agg(anomaly_count=('value','count'),
             earliest_ts=('timestamp','min'),
             window_max_value=('value','max'),
             global_threshold=('global_p95','first'))
    )
    log_high_agg['anomaly_type'] = 'high'
    log_high_agg['severity'] = log_high_agg.apply(
        lambda r: np.inf if r['global_threshold'] == 0 else r['window_max_value'] / r['global_threshold'],
        axis=1
    )
else:
    log_high_agg = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count','earliest_ts','window_max_value','global_threshold','anomaly_type','severity'])

# row_count: low anomalies
log_low = log_merged[is_log_rowcount & (log_merged['value'] <= log_merged['global_p5'])].copy()
if not log_low.empty:
    log_low_agg = (
        log_low
        .groupby(['cmdb_id','log_name'], as_index=False)
        .agg(anomaly_count=('value','count'),
             earliest_ts=('timestamp','min'),
             window_min_value=('value','min'),
             global_threshold=('global_p5','first'))
    )
    log_low_agg['anomaly_type'] = 'low'
    log_low_agg['severity'] = log_low_agg.apply(
        lambda r: np.inf if r['window_min_value'] == 0 else r['global_threshold'] / r['window_min_value'],
        axis=1
    )
    log_low_agg = log_low_agg.rename(columns={'window_min_value':'window_max_value'})
else:
    log_low_agg = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count','earliest_ts','window_max_value','global_threshold','anomaly_type','severity'])

log_anomalies = pd.concat([log_high_agg, log_low_agg], ignore_index=True, sort=False)
log_anomalies_top20 = log_anomalies.sort_values(['anomaly_count','severity'], ascending=[False,False]).head(20)

log_anomalies_top20 = log_anomalies_top20[[
    'cmdb_id','log_name','anomaly_type','anomaly_count','earliest_ts','window_max_value','global_threshold','severity'
]]

# 5) components that had any anomaly in the window: earliest anomaly timestamp per cmdb_id and total anomaly count
all_anoms = pd.concat([
    # normalize column names for concat
    metric_anomalies.assign(item_name=lambda df: df['kpi_name']),
    trace_anomalies.assign(item_name=lambda df: df['trace_name']),
    log_anomalies.assign(item_name=lambda df: df['log_name'])
], sort=False, ignore_index=True)

# keep if any anomalies exist
if not all_anoms.empty:
    components_summary = (
        all_anoms
        .groupby('cmdb_id', as_index=False)
        .agg(total_anomalies=('anomaly_count','sum'),
             earliest_anomaly_ts=('earliest_ts','min'))
        .sort_values(['earliest_anomaly_ts','total_anomalies'], ascending=[True,False])
        .head(20)
    )
else:
    components_summary = pd.DataFrame(columns=['cmdb_id','total_anomalies','earliest_anomaly_ts'])

# 6) sample error_logs rows inside the window (timestamp, cmdb_id, message) up to 20 rows
error_window_samples = error_df[(error_df['timestamp'] >= start_ts) & (error_df['timestamp'] <= end_ts)][['timestamp','cmdb_id','message']].head(20)

# Final compact outputs (do not display full raw series)
metric_window_df.shape, trace_window_df.shape, log_window_df.shape, \
metric_anomalies_top20, trace_anomalies_top20, log_anomalies_top20, \
components_summary, error_window_samples

```
Out[2]:
```
```python
# Compose a concise plain-English summary using existing kernel variables
metric_count, trace_count, log_count = metric_window_df.shape[0], trace_window_df.shape[0], log_window_df.shape[0]

def fmt_anom_rows(df, n=5, name_col='kpi_name'):
    rows = []
    for _, r in df.head(n).iterrows():
        name = r.get(name_col, r.get('trace_name', r.get('log_name', '')))
        rows.append(f"{r['cmdb_id']} | {name} | {r['anomaly_type']} | count={int(r['anomaly_count'])} | severity={float(r['severity']):.3g}")
    return rows

metric_top = fmt_anom_rows(metric_anomalies_top20, 5, 'kpi_name')
trace_top  = fmt_anom_rows(trace_anomalies_top20, 5, 'trace_name')
log_top    = fmt_anom_rows(log_anomalies_top20, 5, 'log_name')

# components summary (top 20 limited already)
components_list = [
    f"{row['cmdb_id']} (total_anomalies={int(row['total_anomalies'])}, earliest_ts={int(row['earliest_anomaly_ts'])})"
    for _, row in components_summary.head(20).iterrows()
]

error_sample_count = error_window_samples.shape[0]
error_sample_examples = [
    f"{int(r['timestamp'])} | {r['cmdb_id']} | {r['message'][:80]}..." for _, r in error_window_samples.head(5).iterrows()
]

summary = (
    "Incident window summary (2021-03-24 20:30 — 21:00 UTC+8):\n\n"
    f"1) Rows in window: metric={metric_count}, trace={trace_count}, log={log_count}.\n\n"
    "2) Metric anomalies (detected if value >= global P95 or <= global P5):\n"
    f"   - Top metric anomaly examples (up to 5):\n     - " + "\n     - ".join(metric_top) + "\n"
    "   - Note: many metric anomalies show global threshold == 0 (severity=inf); investigate thresholds/zero-baseline metrics.\n\n"
    "3) Trace anomalies (duration => high if >=P95; row_count => low if <=P5):\n"
    f"   - Top trace anomaly examples (up to 5):\n     - " + "\n     - ".join(trace_top) + "\n\n"
    "4) Log anomalies (log.error_count => high if >=P95; log.row_count => low if <=P5):\n"
    f"   - Top log anomaly examples (up to 5):\n     - " + "\n     - ".join(log_top) + "\n\n"
    "5) Components with anomalies (earliest anomaly timestamp shown, top 20):\n"
    "   - " + "\n   - ".join(components_list) + "\n\n"
    f"6) error_logs samples inside window: {error_sample_count} rows (showing up to 5 examples):\n"
    "   - " + "\n   - ".join(error_sample_examples) + "\n\n"
    "Notes: Full filtered tables and intermediate anomaly tables remain in kernel variables for follow-up (metric_window_df, trace_window_df, "
    "log_window_df, metric_anomalies_top20, trace_anomalies_top20, log_anomalies_top20, components_summary, error_window_samples)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((29506, 4), (6939, 4), (400, 4),      cmdb_id                                           kpi_name anomaly_type  anomaly_count  earliest_ts  window_max_value  global_threshold  severity
360     MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT         high             31   1616589000               1.0               0.0       inf
361     MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         high             31   1616589000               0.0               0.0       inf
386  Mysql01        Mysql-MySQL_3306_Binlog stmt cache disk use         high             31   1616589000               0.0               0.0       inf
391  Mysql01                  Mysql-MySQL_3306_Com Delete Multi         high             31   1616589000               0.0               0.0       inf
393  Mysql01                 Mysql-MySQL_3306_Com Insert Select         high             31   1616589000               0.0               0.0       inf
394  Mysql01                          Mysql-MySQL_3306_Com Load         high             31   1616589000               0.0               0.0       inf
395  Mysql01                       Mysql-MySQL_3306_Com Replace         high             31   1616589000               0.0               0.0       inf
396  Mysql01                Mysql-MySQL_3306_Com Replace Select         high             31   1616589000               0.0               0.0       inf
399  Mysql01                  Mysql-MySQL_3306_Com Update Multi         high             31   1616589000               0.0               0.0       inf
404  Mysql01          Mysql-MySQL_3306_CurrentSQLMaxRunningTime         high             31   1616589000               0.0               0.0       inf
416  Mysql01                 Mysql-MySQL_3306_Handler Savepoint         high             31   1616589000               0.0               0.0       inf
417  Mysql01        Mysql-MySQL_3306_Handler Savepoint Rollback         high             31   1616589000               0.0               0.0       inf
427  Mysql01     Mysql-MySQL_3306_Innodb buffer pool read ahead         high             31   1616589000               0.0               0.0       inf
428  Mysql01  Mysql-MySQL_3306_Innodb buffer pool read ahead...         high             31   1616589000               0.0               0.0       inf
429  Mysql01  Mysql-MySQL_3306_Innodb buffer pool read ahead...         high             31   1616589000               0.0               0.0       inf
432  Mysql01      Mysql-MySQL_3306_Innodb buffer pool wait free         high             31   1616589000               0.0               0.0       inf
435  Mysql01        Mysql-MySQL_3306_Innodb data pending fsyncs         high             31   1616589000               0.0               0.0       inf
436  Mysql01         Mysql-MySQL_3306_Innodb data pending reads         high             31   1616589000               0.0               0.0       inf
437  Mysql01        Mysql-MySQL_3306_Innodb data pending writes         high             31   1616589000               0.0               0.0       inf
444  Mysql01                  Mysql-MySQL_3306_Innodb log waits         high             31   1616589000               0.0               0.0       inf,       cmdb_id                     trace_name anomaly_type  anomaly_count  earliest_ts  window_max_value  global_threshold   severity
141  dockerB1        trace.self.duration_p95         high              5   1616589060             0.053             0.011   4.818182
173      MG01  trace.from_dockerA2.row_count          low              4   1616589120             1.000             3.000   3.000000
224  dockerA2        trace.to_MG01.row_count          low              4   1616589120             1.000             3.000   3.000000
172      MG01  trace.from_dockerA1.row_count          low              4   1616589000             4.000             6.000   1.500000
219  dockerA1        trace.to_MG01.row_count          low              4   1616589000             4.000             6.000   1.500000
178      MG01    trace.to_dockerA2.row_count          low              3   1616589420             1.000            34.000  34.000000
221  dockerA2      trace.from_MG01.row_count          low              3   1616589420             1.000            34.000  34.000000
163      IG02           trace.self.row_count          low              3   1616589480            24.000           594.000  24.750000
185      MG02  trace.from_dockerA1.row_count          low              3   1616589180             4.000             5.000   1.250000
220  dockerA1        trace.to_MG02.row_count          low              3   1616589180             4.000             5.000   1.250000
171      MG01  trace.from_Tomcat04.row_count          low              3   1616589420            76.000            77.000   1.013158
214  Tomcat04        trace.to_MG01.row_count          low              3   1616589420            76.000            77.000   1.013158
121  dockerA1        trace.self.duration_p95         high              3   1616590200             0.008             0.008   1.000000
159      IG01    trace.to_Tomcat01.row_count          low              2   1616589420             1.000            72.500  72.500000
194  Tomcat01      trace.from_IG01.row_count          low              2   1616589420             1.000            72.500  72.500000
168      MG01  trace.from_Tomcat01.row_count          low              2   1616589480             1.000            65.500  65.500000
197  Tomcat01        trace.to_MG01.row_count          low              2   1616589480             1.000            65.500  65.500000
196  Tomcat01           trace.self.row_count          low              2   1616589300            18.000           974.500  54.138889
213  Tomcat04           trace.self.row_count          low              2   1616589420            19.000           945.500  49.763158
158      IG01           trace.self.row_count          low              2   1616589420            14.000           611.000  43.642857,      cmdb_id         log_name anomaly_type  anomaly_count  earliest_ts  window_max_value  global_threshold   severity
0   Tomcat01  log.error_count         high             31   1616589000               0.0               0.0        inf
1   Tomcat02  log.error_count         high             31   1616589000               0.0               0.0        inf
3   Tomcat04  log.error_count         high             31   1616589000               0.0               0.0        inf
4   apache01  log.error_count         high             31   1616589000              19.0               0.0        inf
2   Tomcat03  log.error_count         high             27   1616589000               0.0               0.0        inf
6   Tomcat01    log.row_count          low              2   1616589300               3.0             146.0  48.666667
10  apache01    log.row_count          low              2   1616589420               7.0             296.5  42.357143
9   Tomcat04    log.row_count          low              2   1616589420               4.0             144.0  36.000000
11  apache02    log.row_count          low              2   1616589480              12.0             310.0  25.833333
7   Tomcat02    log.row_count          low              2   1616589420               6.0             151.5  25.250000
5   apache02  log.error_count         high              2   1616589420              28.0               1.5  18.666667
8   Tomcat03    log.row_count          low              2   1616589240               7.0              56.9   8.128571,           cmdb_id  total_anomalies  earliest_anomaly_ts
4         Mysql01             5609           1616589000
5         Mysql02             5106           1616589000
20       Tomcat02             2692           1616589000
21       Tomcat03             2580           1616589000
22       Tomcat04             2576           1616589000
19       Tomcat01             2434           1616589000
7         Redis02             2299           1616589000
3            MG02             1997           1616589000
2            MG01             1917           1616589000
6         Redis01             1869           1616589000
1            IG02             1156           1616589000
0            IG01             1132           1616589000
24       apache02             1091           1616589000
23       apache01             1074           1616589000
26       dockerA2              265           1616589000
18   ServiceTest9               65           1616589000
8    ServiceTest1               64           1616589000
9   ServiceTest10               64           1616589000
10  ServiceTest11               64           1616589000
11   ServiceTest2               64           1616589000,      timestamp   cmdb_id                                            message
0   1616589452  apache01  IPAddress "POST /UOCP/person/ServiceTest1.json...
1   1616589452  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
2   1616589452  apache01  IPAddress "POST /UOCP/person/ServiceTest1.json...
3   1616589454  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
4   1616589455  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
5   1616589455  apache01  IPAddress "POST /UOCP/person/ServiceTest1.json...
6   1616589455  apache02  IPAddress "POST /UOCP/person/ServiceTest3.json...
7   1616589455  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
8   1616589455  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
9   1616589456  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...
10  1616589456  apache01  IPAddress "POST /UOCP/person/ServiceTest1.json...
11  1616589456  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
12  1616589456  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...
13  1616589457  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
14  1616589457  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
15  1616589458  apache01  IPAddress "POST /UOCP/person/ServiceTest1.json...
16  1616589458  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...
17  1616589458  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
18  1616589459  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
19  1616589459  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables: metric_window_df, trace_window_df, log_window_df, error_df
# Candidate components and narrow time range
candidates = ['apache01','apache02','IG01','Tomcat01','Tomcat04','MG01','MG02']
start_narrow = 1616589400
end_narrow   = 1616589480

# 1) Filter each source to the narrow range and candidate components
metric_narrow = metric_window_df[(metric_window_df['timestamp'] >= start_narrow) & (metric_window_df['timestamp'] <= end_narrow) & (metric_window_df['cmdb_id'].isin(candidates))].copy()
trace_narrow  = trace_window_df[(trace_window_df['timestamp']  >= start_narrow) & (trace_window_df['timestamp']  <= end_narrow) & (trace_window_df['cmdb_id'].isin(candidates))].copy()
log_narrow    = log_window_df[(log_window_df['timestamp']    >= start_narrow) & (log_window_df['timestamp']    <= end_narrow) & (log_window_df['cmdb_id'].isin(candidates))].copy()
error_narrow  = error_df[(error_df['timestamp'] >= start_narrow) & (error_df['timestamp'] <= end_narrow) & (error_df['cmdb_id'].isin(candidates))].copy()

# Aggregated counts per component per source (limit to the 7 components)
counts = []
for c in candidates:
    counts.append({
        'cmdb_id': c,
        'metric_count': int(metric_narrow[metric_narrow['cmdb_id'] == c].shape[0]),
        'trace_count' : int(trace_narrow[trace_narrow['cmdb_id'] == c].shape[0]),
        'log_count'   : int(log_narrow[log_narrow['cmdb_id'] == c].shape[0]),
        'error_count' : int(error_narrow[error_narrow['cmdb_id'] == c].shape[0])
    })
counts_df = pd.DataFrame(counts)

# 2) Up to 200 time-ordered sample rows across sources and components
def build_samples(df, source_label):
    if source_label == 'error':
        # message truncated to 120 chars as requested, value blank
        s = df[['timestamp','cmdb_id','message']].copy()
        s = s.rename(columns={'message':'name'})
        s['name'] = s['name'].astype(str).str.slice(0,120)
        s['value'] = ''
    else:
        name_col = 'kpi_name' if source_label == 'metric' else ('trace_name' if source_label == 'trace' else 'log_name')
        s = df[['timestamp','cmdb_id',name_col,'value']].copy()
        s = s.rename(columns={name_col:'name'})
        # ensure numeric values; keep as-is
        s['value'] = s['value']
    s['source'] = source_label
    return s[['timestamp','source','cmdb_id','name','value']]

samples_metric = build_samples(metric_narrow, 'metric')
samples_trace  = build_samples(trace_narrow, 'trace')
samples_log    = build_samples(log_narrow, 'log')
samples_error  = build_samples(error_narrow, 'error')

samples_all = pd.concat([samples_metric, samples_trace, samples_log, samples_error], ignore_index=True, sort=False)
samples_all = samples_all.sort_values('timestamp').reset_index(drop=True)
samples_ordered = samples_all.head(200)  # up to 200 rows time-ordered

# 3) For each candidate component, extract short time-series for key KPIs if present:
# trace.self.row_count, any trace.to_*.row_count and trace.from_*.row_count, trace.*.duration_p95,
# log.error_count, log.row_count.
# We'll collect trace KPIs first
trace_kpis_mask = (
    trace_narrow['trace_name'].str.contains(r'(^trace\.self\.row_count$)|(\.row_count$)|(\.duration_p95$)', regex=True, na=False)
)
trace_kpi_series = trace_narrow[trace_kpis_mask].copy()
# Keep only relevant columns and limit per (cmdb_id, trace_name) to 10 rows sorted by timestamp
trace_kpi_series = trace_kpi_series.sort_values(['cmdb_id','trace_name','timestamp'])
trace_kpi_series['rank'] = trace_kpi_series.groupby(['cmdb_id','trace_name']).cumcount() + 1
trace_kpi_series_limited = trace_kpi_series[trace_kpi_series['rank'] <= 10][['timestamp','cmdb_id','trace_name','value']].rename(columns={'trace_name':'name'})

# Logs KPIs
log_kpis_mask = log_narrow['log_name'].isin(['log.error_count','log.row_count'])
log_kpi_series = log_narrow[log_kpis_mask].copy().sort_values(['cmdb_id','log_name','timestamp'])
log_kpi_series['rank'] = log_kpi_series.groupby(['cmdb_id','log_name']).cumcount() + 1
log_kpi_series_limited = log_kpi_series[log_kpi_series['rank'] <= 10][['timestamp','cmdb_id','log_name','value']].rename(columns={'log_name':'name'})

# Combine KPI series (trace + log). Metrics are not requested for this specific KPI list.
kpi_series = pd.concat([trace_kpi_series_limited, log_kpi_series_limited], ignore_index=True, sort=False)
kpi_series = kpi_series.sort_values(['cmdb_id','name','timestamp']).reset_index(drop=True)

# 4) Earliest timestamp in this range where any of the key KPIs crosses its global threshold.
# Use precomputed global thresholds if available (trace_thresholds, log_thresholds). If not present, compute them now.
try:
    _ = trace_thresholds
except NameError:
    trace_thresholds = (
        trace_df
        .groupby(['cmdb_id','trace_name'], as_index=False)['value']
        .agg(global_p95=lambda x: x.quantile(0.95),
             global_p5 =lambda x: x.quantile(0.05))
    )
try:
    _ = log_thresholds
except NameError:
    log_thresholds = (
        log_df
        .groupby(['cmdb_id','log_name'], as_index=False)['value']
        .agg(global_p95=lambda x: x.quantile(0.95),
             global_p5 =lambda x: x.quantile(0.05))
    )

# For trace row_count KPIs: breach if value <= global_p5
trace_breach_candidates = kpi_series[kpi_series['name'].str.contains(r'\.row_count$', na=False)].merge(
    trace_thresholds, left_on=['cmdb_id','name'], right_on=['cmdb_id','trace_name'], how='left'
)
trace_breach_candidates['breach'] = trace_breach_candidates['value'] <= trace_breach_candidates['global_p5']

# For log.error_count: breach if value >= global_p95
log_breach_candidates = log_kpi_series_limited.merge(
    log_thresholds, left_on=['cmdb_id','name'], right_on=['cmdb_id','log_name'], how='left'
)
log_breach_candidates['breach'] = log_breach_candidates['value'] >= log_breach_candidates['global_p95']

# Combine breaches and find earliest per component
trace_breaches = trace_breach_candidates[trace_breach_candidates['breach']].copy()
trace_breaches = trace_breaches[['cmdb_id','name','timestamp']]

log_breaches = log_breach_candidates[log_breach_candidates['breach']][['cmdb_id','name','timestamp']].copy()

all_breaches = pd.concat([trace_breaches, log_breaches], ignore_index=True, sort=False)
if all_breaches.empty:
    earliest_breaches = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_breach_ts'])
else:
    earliest_breaches = (
        all_breaches
        .groupby('cmdb_id', as_index=False)
        .agg(kpi_name=('name','first'), earliest_breach_ts=('timestamp','min'))
    )
    # Keep only candidate components and limit to one earliest breach per component
    earliest_breaches = earliest_breaches[earliest_breaches['cmdb_id'].isin(candidates)].sort_values('earliest_breach_ts').reset_index(drop=True)

# Store filtered results in kernel variables for next steps
metric_narrow_df = metric_narrow
trace_narrow_df  = trace_narrow
log_narrow_df    = log_narrow
error_narrow_df  = error_narrow

# Final compact outputs (bounded): counts_df, samples_ordered (<=200), kpi_series (per-component KPI rows limited), earliest_breaches
counts_df, samples_ordered, kpi_series.head(200), earliest_breaches.head(20)

```
Out[3]:
```
```python
# Compose a concise plain-English summary from the kernel variables
counts = counts_df.copy()
samples_n = samples_ordered.shape[0]
sample_examples = []
for _, r in samples_ordered.head(3).iterrows():
    val = r['value']
    val_str = '' if pd.isna(val) or val=='' else str(val)
    sample_examples.append(f"{int(r['timestamp'])} | {r['source']} | {r['cmdb_id']} | {r['name'][:60]} | {val_str}")

# KPI series presence counts per candidate component
kpi_counts = kpi_series.groupby('cmdb_id').size().to_dict()

# Earliest breaches per component (if any)
breaches = {}
for _, r in earliest_breaches.iterrows():
    breaches[r['cmdb_id']] = (r['kpi_name'], int(r['earliest_breach_ts']))

lines = []
lines.append("Narrow-range telemetry summary (1616589400 — 1616589480):")
lines.append("")
lines.append("1) Row counts per component and source (metric / trace / log / error):")
for _, r in counts.iterrows():
    lines.append(f"   - {r['cmdb_id']}: metric={int(r['metric_count'])}, trace={int(r['trace_count'])}, log={int(r['log_count'])}, error={int(r['error_count'])}")

lines.append("")
lines.append(f"2) Time-ordered samples returned: {samples_n} rows (up to 200). Examples:")
for ex in sample_examples:
    lines.append(f"   - {ex}")

lines.append("")
lines.append("3) Presence of key KPIs (trace.row_count, trace.*.duration_p95, log.error_count, log.row_count):")
for c in ['apache01','apache02','IG01','Tomcat01','Tomcat04','MG01','MG02']:
    kc = kpi_counts.get(c, 0)
    lines.append(f"   - {c}: {kc} matching KPI rows in range (showing up to 10 per KPI).")

lines.append("")
lines.append("4) Earliest global-threshold breach (trace.row_count low OR log.error_count high) per component (one earliest breach shown):")
if breaches:
    for c in ['apache01','apache02','IG01','Tomcat01','Tomcat04','MG01','MG02']:
        if c in breaches:
            kpi, ts = breaches[c]
            lines.append(f"   - {c}: {kpi} breached at timestamp {ts}")
        else:
            lines.append(f"   - {c}: no breach of the checked KPIs in this range")
else:
    lines.append("   - No breaches detected for the checked KPIs in this range.")

lines.append("")
lines.append("Notes: Full narrow-range tables and intermediate results are kept in kernel variables for follow-up:")
lines.append("  metric_narrow_df, trace_narrow_df, log_narrow_df, error_narrow_df, samples_ordered, kpi_series, earliest_breaches")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  metric_count  trace_count  log_count  error_count
0  apache01            88            0          4           19
1  apache02            83            0          4           28
2      IG01            64           24          0            0
3  Tomcat01           144           30          4            0
4  Tomcat04           121           30          5            0
5      MG01           102           72          0            0
6      MG02           100           72          0            0,       timestamp  source   cmdb_id                                               name       value
0    1616589420  metric  apache02         OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity     19.0136
1    1616589420  metric  apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvg...        3.37
2    1616589420  metric  apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...         7.0
3    1616589420  metric  apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps         0.0
4    1616589420  metric  apache02             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem      2135.0
..          ...     ...       ...                                                ...         ...
195  1616589420  metric      IG01          JVM-Operating System_7778_JVM_JVM_CPULoad      0.2497
196  1616589420  metric      IG01                    JVM-Runtime_7778_JVM_JVM_Uptime   1668465.0
197  1616589420  metric      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  58869320.0
198  1616589420  metric      MG01          JVM-Operating System_7779_JVM_JVM_CPULoad      1.0075
199  1616589420  metric      MG01                    JVM-Runtime_7779_JVM_JVM_Uptime    558656.0

[200 rows x 5 columns],       timestamp   cmdb_id                            name      value
0    1616589420      IG01         trace.self.duration_p95   15.55545
1    1616589480      IG01         trace.self.duration_p95   15.58410
2    1616589420      IG01            trace.self.row_count  590.00000
3    1616589480      IG01            trace.self.row_count  670.00000
4    1616589420      IG01  trace.to_Tomcat01.duration_p95    1.07980
..          ...       ...                             ...        ...
163  1616589480  apache01                   log.row_count  329.00000
164  1616589420  apache02                 log.error_count   28.00000
165  1616589480  apache02                 log.error_count    0.00000
166  1616589420  apache02                   log.row_count  320.00000
167  1616589480  apache02                   log.row_count  303.00000

[168 rows x 4 columns],     cmdb_id                       kpi_name  earliest_breach_ts
0      IG01           trace.self.row_count          1616589420
1      MG01  trace.from_Tomcat01.row_count          1616589420
2      MG02  trace.from_Tomcat01.row_count          1616589420
3  Tomcat01      trace.from_IG01.row_count          1616589420
4  Tomcat04      trace.from_IG01.row_count          1616589420
5  apache01                log.error_count          1616589420
6  apache02                log.error_count          1616589420)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import re
import pytz

# Reuse existing kernel variables: metric_df
tz = pytz.timezone('Asia/Shanghai')

# Candidate components
candidates = ['apache01','apache02','IG01','Tomcat01','Tomcat04','MG01','MG02']

# 1) select kpi_name values containing any of the given substrings (case-sensitive)
substrs = ['JVM','MEMORY','CPU','NETWORK','DSK','DISK','FILESYSTEM']
pattern = '|'.join(re.escape(s) for s in substrs)

metric_kpi_candidates = (
    metric_df[
        metric_df['cmdb_id'].isin(candidates) &
        metric_df['kpi_name'].str.contains(pattern, regex=True, na=False)
    ][['cmdb_id','kpi_name']]
    .drop_duplicates()
    .reset_index(drop=True)
    .head(200)   # limit total rows to 200
)

# 2) global aggregates for the selected (cmdb_id, kpi_name) pairs using full series
# First get the full list of candidate pairs (not limited by the earlier head(200) if many; 
# but comply with user's limit by using the pairs present in metric_kpi_candidates)
candidate_pairs = metric_kpi_candidates[['cmdb_id','kpi_name']].drop_duplicates()

# Merge to full metric_df to compute global stats only for selected pairs
metric_selected_full = metric_df.merge(candidate_pairs, on=['cmdb_id','kpi_name'], how='inner')

metric_kpi_candidates_global_stats = (
    metric_selected_full
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(count='count',
         p50=lambda x: x.quantile(0.5),
         p95=lambda x: x.quantile(0.95),
         max_value='max')
    .sort_values('count', ascending=False)
    .head(50)   # return up to top 50 rows sorted by count desc
)

# 3) extract metric values within the narrow time range 1616589400 — 1616589480
start_narrow = 1616589400
end_narrow   = 1616589480

metric_kpi_candidates_window_samples = (
    metric_df[
        (metric_df['timestamp'] >= start_narrow) &
        (metric_df['timestamp'] <= end_narrow) &
        metric_df['cmdb_id'].isin(candidate_pairs['cmdb_id']) &
        metric_df['kpi_name'].isin(candidate_pairs['kpi_name'])
    ][['timestamp','cmdb_id','kpi_name','value']]
    .sort_values(['cmdb_id','kpi_name','timestamp'])
)

# limit to up to 10 rows per (cmdb_id,kpi_name)
metric_kpi_candidates_window_samples = (
    metric_kpi_candidates_window_samples
    .groupby(['cmdb_id','kpi_name'], as_index=False, group_keys=False)
    .apply(lambda df: df.head(10))
    .reset_index(drop=True)
)

# enforce total returned rows cap of 200
metric_kpi_candidates_window_samples = metric_kpi_candidates_window_samples.head(200)

# Display the three compact tables (stored in kernel variables for further steps)
metric_kpi_candidates, metric_kpi_candidates_global_stats, metric_kpi_candidates_window_samples

```
Out[4]:
```
```python
# Compose a concise plain-English summary based on existing kernel variables:
# metric_kpi_candidates, metric_kpi_candidates_global_stats, metric_kpi_candidates_window_samples

import pandas as pd
import numpy as np

# Helpers to safely fetch values
def find_global(cmdb, pattern):
    df = metric_kpi_candidates_global_stats
    sel = df[(df['cmdb_id'] == cmdb) & df['kpi_name'].str.contains(pattern, na=False)]
    return sel.sort_values('count', ascending=False).reset_index(drop=True)

def find_window(cmdb, pattern):
    df = metric_kpi_candidates_window_samples
    sel = df[(df['cmdb_id'] == cmdb) & df['kpi_name'].str.contains(pattern, na=False)]
    return sel.sort_values('timestamp').reset_index(drop=True)

# Candidate list
candidates = ['apache01','apache02','IG01','Tomcat01','Tomcat04','MG01','MG02']

# 1) counts of resource KPIs found per component
counts = metric_kpi_candidates.groupby('cmdb_id').size().reindex(candidates, fill_value=0).to_dict()

# 2) extract notable global aggregates and window samples for quick findings
notes = []

# IG01: look for JVM HeapMemoryUsed / HeapMemoryUsage
g_ig_heap = find_global('IG01', 'HeapMemoryUsed|HeapMemoryUsage')
w_ig_heap = find_window('IG01', 'HeapMemoryUsed|HeapMemoryUsage')

if not g_ig_heap.empty:
    # pick first match
    r = g_ig_heap.iloc[0]
    notes.append(f"IG01 JVM: global count={int(r['count'])}, P50={r['p50']:.3g}, P95={r['p95']:.3g}, max={r['max_value']:.3g}.")
    if not w_ig_heap.empty:
        # show earliest window sample for this KPI
        w0 = w_ig_heap.iloc[0]
        notes.append(f"  At {int(w0['timestamp'])} IG01 {w0['kpi_name']} = {w0['value']:.3g} (within narrow range).")

# MG02: memory used percent
g_mg02_mem = find_global('MG02', 'MEMUsedMemPerc|MEMORY_MEMUsedMemPerc|MEMORY_MEMUsed')
w_mg02_mem = find_window('MG02', 'MEMUsedMemPerc|MEMORY_MEMUsedMemPerc|MEMORY_MEMUsed')

if not g_mg02_mem.empty:
    r = g_mg02_mem.iloc[0]
    notes.append(f"MG02 Memory: global count={int(r['count'])}, P50={r['p50']:.3g}, P95={r['p95']:.3g}, max={r['max_value']:.3g}.")
    if not w_mg02_mem.empty:
        w0 = w_mg02_mem.iloc[0]
        notes.append(f"  At {int(w0['timestamp'])} MG02 {w0['kpi_name']} = {w0['value']:.3g}.")

# apache01 / Tomcat01: CPU utilization
g_ap_cpu = find_global('apache01', 'CPU|CPUCpuUtil|CPU_CPU')
w_ap_cpu = find_window('apache01', 'CPU|CPUCpuUtil|CPU_CPU')
if not g_ap_cpu.empty:
    r = g_ap_cpu.iloc[0]
    notes.append(f"apache01 CPU: global count={int(r['count'])}, P50={r['p50']:.3g}, P95={r['p95']:.3g}, max={r['max_value']:.3g}.")
    if not w_ap_cpu.empty:
        w0 = w_ap_cpu.iloc[0]
        notes.append(f"  At {int(w0['timestamp'])} apache01 {w0['kpi_name']} = {w0['value']:.3g}.")

g_tc1_cpu = find_global('Tomcat01', 'CPU|CPUCpuUtil|CPU_CPU')
w_tc1_cpu = find_window('Tomcat01', 'CPU|CPUCpuUtil|CPU_CPU')
if not g_tc1_cpu.empty:
    r = g_tc1_cpu.iloc[0]
    notes.append(f"Tomcat01 CPU: global count={int(r['count'])}, P50={r['p50']:.3g}, P95={r['p95']:.3g}, max={r['max_value']:.3g}.")
    if not w_tc1_cpu.empty:
        w0 = w_tc1_cpu.iloc[0]
        notes.append(f"  At {int(w0['timestamp'])} Tomcat01 {w0['kpi_name']} = {w0['value']:.3g}.")

# NETWORK / DSK / FILESYSTEM for components: show any top hits
for c in candidates:
    g_net = find_global(c, 'NETWORK|NET|TotalTcpConn|NETIn|NETOut')
    g_dsk = find_global(c, 'DSK|DISK|LOCALDISK|FILESYSTEM|FSCapacity|FSUsed')
    if not g_net.empty:
        r = g_net.iloc[0]
        notes.append(f"{c} Network-related KPI example: {r['kpi_name']} P95={r['p95']:.3g}, max={r['max_value']:.3g}.")
    if not g_dsk.empty:
        r = g_dsk.iloc[0]
        notes.append(f"{c} Disk/FS KPI example: {r['kpi_name']} P95={r['p95']:.3g}, max={r['max_value']:.3g}.")

# Build final summary string
lines = []
lines.append("Resource KPI scan for candidates (substrings: JVM, MEMORY, CPU, NETWORK, DSK, DISK, FILESYSTEM):")
lines.append("")
lines.append("1) Count of matching resource KPI names per component:")
for c in candidates:
    lines.append(f"   - {c}: {int(counts.get(c,0))} unique KPI names")

lines.append("")
lines.append("2) Notable global aggregates and narrow-range observations:")
if notes:
    for n in notes:
        lines.append("   - " + n)
else:
    lines.append("   - No notable resource KPI global stats found for the candidates.")

lines.append("")
lines.append("3) Quick guidance:")
lines.append("   - IG01 shows JVM HeapMemoryUsed near its P95 at the incident time (see above); investigate JVM heap pressure for IG01.")
lines.append("   - MG02 shows very high MEMUsed percentage (saturated at ~91% in global stats); check host memory pressure on MG02.")
lines.append("   - apache01 and Tomcat01 have moderate CPU util (around ~25-27% P95), not obviously CPU-saturated from these metrics.")
lines.append("   - Check network/disk KPIs listed above per component if further evidence is needed.")
lines.append("")
lines.append("Stored variables for follow-up analysis:")
lines.append("  metric_kpi_candidates  (unique candidate KPI names)")
lines.append("  metric_kpi_candidates_global_stats  (global count/P50/P95/max)")
lines.append("  metric_kpi_candidates_window_samples (narrow-range samples up to 200 rows)")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                         kpi_name
0      IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryMax
1      IG01       JVM-Memory_7778_JVM_Memory_HeapMemoryUsage
2      IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed
3      IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed
4      IG01                  JVM-Runtime_7778_JVM_JVM_Uptime
..      ...                                              ...
195    IG01   OSLinux-OSLinux_FILESYSTEM_-tomcat_FSUsedSpace
196    IG01     OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc
197    IG01          OSLinux-OSLinux_NETWORK_ens160_NETInErr
198    IG01  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec
199    IG01         OSLinux-OSLinux_NETWORK_ens160_NETOutErr

[200 rows x 2 columns],       cmdb_id                                         kpi_name  count           p50           p95     max_value
0        IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09
2        IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  4.690916e+08  8.668898e+08  8.686124e+08
3        IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.848308e+07  5.848308e+07  5.848308e+07
6        IG01         JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.100000e+01  7.500000e+01  7.600000e+01
46       MG01                       OSLinux-CPU_CPU_CPUSysTime     31  6.819000e-01  8.461540e-01  8.981000e-01
..        ...                                              ...    ...           ...           ...           ...
97   Tomcat01                       OSLinux-CPU_CPU_CPUCpuUtil     21  2.629720e+01  2.666280e+01  2.670240e+01
89       MG02  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum     21  1.947000e+03  2.432000e+03  2.583000e+03
163  apache01                       OSLinux-CPU_CPU_CPUCpuUtil     21  2.552620e+01  2.566030e+01  2.573800e+01
179  apache01            OSLinux-OSLinux_MEMORY_MEMORY_UserMem     21  2.466000e+03  2.467000e+03  2.467000e+03
85       MG02     OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc     21  9.100000e+01  9.100000e+01  9.100000e+01

[50 rows x 6 columns],       timestamp   cmdb_id                                           kpi_name         value
0    1616589420      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1    1616589480      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
2    1616589420      IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.689980e+01
3    1616589420      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.670007e+08
4    1616589480      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  1.141281e+08
..          ...       ...                                                ...           ...
195  1616589480  Tomcat01  OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCa...  4.268120e-01
196  1616589480  Tomcat01  OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSIn...  4.196000e-03
197  1616589480  Tomcat01     OSLinux-OSLinux_FILESYSTEM_----home_FSCapacity  3.331767e+00
198  1616589480  Tomcat01  OSLinux-OSLinux_FILESYSTEM_----home_FSInodeUse...  2.201840e-01
199  1616589480  Tomcat01      OSLinux-OSLinux_FILESYSTEM_----tmp_FSCapacity  9.759760e-01

[200 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df from kernel; ensure timezone var exists per rules
tz = pytz.timezone('Asia/Shanghai')

# 1) Identify IG01 heap KPI names in full series and compute global P95 per kpi_name
ig_mask = (metric_df['cmdb_id'] == 'IG01') & metric_df['kpi_name'].str.contains('HeapMemoryUsed|HeapMemoryUsage', regex=True, na=False)
metric_ig_heap_kpis = metric_df[ig_mask][['kpi_name']].drop_duplicates().reset_index(drop=True)

# Compute global P95 per (cmdb_id,kpi_name) using full metric_df (rule 9)
metric_ig_heap_thresholds = (
    metric_df[metric_df['cmdb_id'] == 'IG01']
    .groupby('kpi_name', as_index=True)['value']
    .quantile(0.95)
    .rename('global_p95')
    .reset_index()
)

# Keep only the heap KPIs
metric_ig_heap_thresholds = metric_ig_heap_thresholds[metric_ig_heap_thresholds['kpi_name'].isin(metric_ig_heap_kpis['kpi_name'])].reset_index(drop=True)

# 2) Filter metric_df to the full incident window and IG01 heap KPIs
start_incident = 1616589000
end_incident   = 1616590800

metric_ig_window = metric_df[
    (metric_df['cmdb_id'] == 'IG01') &
    (metric_df['timestamp'] >= start_incident) &
    (metric_df['timestamp'] <= end_incident) &
    (metric_df['kpi_name'].isin(metric_ig_heap_thresholds['kpi_name']))
].copy()

# Merge global threshold into the windowed data
metric_ig_window = metric_ig_window.merge(metric_ig_heap_thresholds, on='kpi_name', how='left')

# Detect consecutive anomaly segments where value >= global_p95
segments = []
for kpi, sub in metric_ig_window.groupby('kpi_name'):
    dfk = sub.sort_values('timestamp').reset_index(drop=True)
    # Mark anomalies using global P95 (which was computed on full series)
    dfk['is_anom'] = dfk['value'] >= dfk['global_p95']
    # Keep only anomalous rows for segment detection
    anom = dfk[dfk['is_anom']].copy()
    if anom.empty:
        continue
    # compute time diff between consecutive anomalous rows
    anom['ts_diff'] = anom['timestamp'].diff().fillna(99999).astype(int)
    # new segment when diff != 60 (one minute)
    anom['segment_id'] = (anom['ts_diff'] != 60).cumsum()
    # aggregate segments
    seg_agg = anom.groupby('segment_id').agg(
        kpi_name = ('kpi_name','first'),
        segment_start_ts = ('timestamp','min'),
        segment_end_ts = ('timestamp','max'),
        segment_length_in_points = ('timestamp','count'),
        segment_max_value = ('value','max'),
        global_P95 = ('global_p95','first')
    ).reset_index(drop=True)
    # compute severity and filter length >=2
    if not seg_agg.empty:
        seg_agg['severity'] = seg_agg.apply(
            lambda r: (np.inf if r['global_P95'] == 0 else (r['segment_max_value'] / r['global_P95'])),
            axis=1
        )
        seg_agg = seg_agg[seg_agg['segment_length_in_points'] >= 2]
        for _, row in seg_agg.iterrows():
            segments.append(row.to_dict())

# Build DataFrame of segments and sort/limit
if segments:
    metric_ig_heap_faults = pd.DataFrame(segments)
    metric_ig_heap_faults = metric_ig_heap_faults[
        ['kpi_name','segment_start_ts','segment_end_ts','segment_length_in_points','segment_max_value','global_P95','severity']
    ].sort_values(['severity','segment_length_in_points'], ascending=[False,False]).reset_index(drop=True).head(10)
else:
    metric_ig_heap_faults = pd.DataFrame(columns=['kpi_name','segment_start_ts','segment_end_ts','segment_length_in_points','segment_max_value','global_P95','severity'])

# 3) earliest segment_start_ts across all IG01 heap KPIs (or None)
if not metric_ig_heap_faults.empty:
    ig_heap_earliest_breach = int(metric_ig_heap_faults['segment_start_ts'].min())
else:
    ig_heap_earliest_breach = None

# Return compact outputs (stored in kernel variables as requested)
metric_ig_heap_faults, ig_heap_earliest_breach

```
Out[5]:
```
```python
# Plain-English summary string based on kernel variables metric_ig_heap_faults and ig_heap_earliest_breach
metric_ig_heap_faults, ig_heap_earliest_breach
summary = (
    "IG01 JVM heap analysis (plain English):\n\n"
    "1) KPIs examined: JVM heap KPIs containing 'HeapMemoryUsed' or 'HeapMemoryUsage' were checked using global P95 thresholds computed from the full series.\n\n"
    "2) Detected consecutive anomaly segments (value >= global P95, length >= 2):\n"
    "   - One segment found:\n"
    "     • kpi_name: JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed\n"
    "     • segment_start_ts: 1616589000\n"
    "     • segment_end_ts:   1616590800\n"
    "     • segment_length:   31 points (continuous per-minute coverage across the incident)\n"
    "     • segment_max_value: 58483080.0\n"
    "     • global_P95:         58483080.0\n"
    "     • severity (max / P95): 1.0\n\n"
    "3) Earliest breach timestamp across IG01 heap KPIs: 1616589000\n\n"
    "4) Interpretation: The NoHeapMemoryUsed metric was at its historical 95th-percentile for the entire incident window (sustained, not just an isolated spike). "
    "Because severity == 1.0 (max equals global P95), the metric reached the P95 baseline but did not exceed it by the observed maximum. "
    "No other IG01 JVM heap KPI produced multi-point (>=2) breaches of global P95 in the incident window.\n\n"
    "5) Suggested next steps: investigate sustained JVM non-heap usage and related GC or classloader activity on IG01, and check HeapMemoryUsed/HeapMemoryUsage time-series and application logs for corroborating signs of heap pressure or GC stalls."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                      kpi_name  segment_start_ts  segment_end_ts  segment_length_in_points  segment_max_value  global_P95  severity
0  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed        1616589000      1616590800                        31         58483080.0  58483080.0       1.0, 1616589000)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import re
import pytz

# reuse metric_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Candidate: IG01 only
cmdb = 'IG01'

# 1) find matching kpi_name values (case-sensitive substrings as provided)
substrs = [
    'JVM_JVM_CPULoad','JVM_JVM','gc','GC','gc_pause','Thread','CPU','CPUCpuUtil',
    'OSLinux-OSLinux_CPU','NET','NETInErr','NETOutErr','NETKB','NETWORK',
    'MEMORY','MEMUsed','HeapMemoryUsed','HeapMemoryUsage','NoHeapMemoryUsed',
    'FSCapacity','DSK','DISK'
]
# Build regex: any of substrings (some contain special chars; escape)
pattern = '(' + '|'.join(re.escape(s) for s in substrs) + ')'

metric_ig_candidates = (
    metric_df[
        (metric_df['cmdb_id'] == cmdb) &
        metric_df['kpi_name'].str.contains(pattern, regex=True, na=False)
    ][['cmdb_id','kpi_name']]
    .drop_duplicates()
    .reset_index(drop=True)
    .head(200)  # limit total rows to 200
)

# 2) global aggregates for those IG01 kpis using full series (compute thresholds BEFORE any filtering)
candidate_kpis = metric_ig_candidates['kpi_name'].unique().tolist()

metric_ig_selected_full = metric_df[
    (metric_df['cmdb_id'] == cmdb) &
    (metric_df['kpi_name'].isin(candidate_kpis))
].copy()

metric_ig_global_stats = (
    metric_ig_selected_full
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(count='count',
         p50=lambda x: x.quantile(0.5),
         p95=lambda x: x.quantile(0.95),
         max_value='max')
    .sort_values('p95', ascending=False)
    .head(50)  # up to top 50 rows sorted by P95 desc
    .reset_index(drop=True)
)

# 3) windowed samples for narrow window 1616589400 — 1616589480, up to 10 rows per (cmdb_id,kpi_name), total <=200
start_narrow = 1616589400
end_narrow   = 1616589480

metric_ig_window_samples = metric_ig_selected_full[
    (metric_ig_selected_full['timestamp'] >= start_narrow) &
    (metric_ig_selected_full['timestamp'] <= end_narrow)
][['timestamp','cmdb_id','kpi_name','value']].sort_values(['kpi_name','timestamp'])

# limit to up to 10 rows per kpi
metric_ig_window_samples = (
    metric_ig_window_samples
    .groupby(['cmdb_id','kpi_name'], as_index=False, group_keys=False)
    .apply(lambda df: df.head(10))
    .reset_index(drop=True)
).head(200)  # enforce total cap

# 4) detect consecutive segments in full incident window 1616589000—1616590800 where value >= global P95
start_incident = 1616589000
end_incident   = 1616590800

# Prepare global p95 lookup
global_p95_lookup = metric_ig_global_stats.set_index('kpi_name')['p95'].to_dict()

# Filter incident window series for IG01 and candidate kpis
metric_ig_incident = metric_ig_selected_full[
    (metric_ig_selected_full['timestamp'] >= start_incident) &
    (metric_ig_selected_full['timestamp'] <= end_incident)
].copy().sort_values(['kpi_name','timestamp']).reset_index(drop=True)

flagged = []
for kpi, sub in metric_ig_incident.groupby('kpi_name'):
    gp95 = global_p95_lookup.get(kpi, np.nan)
    if np.isnan(gp95):
        continue
    dfk = sub[['timestamp','value']].copy().sort_values('timestamp').reset_index(drop=True)
    dfk['is_anom'] = dfk['value'] >= gp95
    if not dfk['is_anom'].any():
        continue
    # find consecutive runs among anomalous rows: require timestamp diff == 60 for adjacency
    anom = dfk[dfk['is_anom']].copy()
    if anom.empty:
        continue
    anom['ts_diff'] = anom['timestamp'].diff().fillna(99999).astype(int)
    anom['segment_id'] = (anom['ts_diff'] != 60).cumsum()
    segs = anom.groupby('segment_id').agg(
        segment_start_ts=('timestamp','min'),
        segment_end_ts=('timestamp','max'),
        segment_length_in_points=('timestamp','count'),
        segment_max_value=('value','max')
    ).reset_index(drop=True)
    # keep only segments length >=2
    segs = segs[segs['segment_length_in_points'] >= 2]
    if segs.empty:
        continue
    segs['kpi_name'] = kpi
    segs['global_P95'] = gp95
    segs['severity'] = segs['segment_max_value'] / segs['global_P95'].replace({0:np.nan})
    for _, r in segs.iterrows():
        flagged.append({
            'kpi_name': r['kpi_name'],
            'earliest_breach_ts': int(r['segment_start_ts']),
            'segment_length': int(r['segment_length_in_points']),
            'segment_max_value': float(r['segment_max_value']),
            'global_P95': float(r['global_P95']),
            'severity': float(r['severity']) if not np.isnan(r['severity']) else np.inf
        })

if flagged:
    metric_ig_flagged_faults = pd.DataFrame(flagged)
    metric_ig_flagged_faults = metric_ig_flagged_faults.sort_values(['severity','segment_length'], ascending=[False,False]).head(20).reset_index(drop=True)
else:
    metric_ig_flagged_faults = pd.DataFrame(columns=['kpi_name','earliest_breach_ts','segment_length','segment_max_value','global_P95','severity'])

# Store variables as requested and display compact tables
metric_ig_candidates = metric_ig_candidates
metric_ig_global_stats = metric_ig_global_stats
metric_ig_window_samples = metric_ig_window_samples

metric_ig_candidates, metric_ig_global_stats, metric_ig_window_samples, metric_ig_flagged_faults.head(20)

```
Out[6]:
```
Summary of IG01 resource KPI analysis (compact):

- Scope and data:
  - 57 IG01 KPI names matched the resource substrings (JVM, MEMORY, CPU, NETWORK, DSK/DISK, FILESYSTEM, etc.).
  - Global aggregates (full series) computed (count, P50, P95, max) for each matched KPI.
  - Narrow-window samples (1616589400–1616589480) returned 49 metric rows for IG01.

- Key global observations (examples from top-by-P95):
  - JVM-Memory_..._HeapMemoryUsed: P95 ≈ 8.6689e+08 (count 31); max ≈ 8.6861e+08.
  - JVM-Memory_..._NoHeapMemoryUsed: P95 = 5.848308e+07 (count 31); max = 5.848308e+07.
  - OSLinux MEMORY (UserMem / MEMUsedMemPerc) show high memory usage (P95 values indicate host memory heavily used).
  - Network and disk KPIs exist but their P95 values are not the top outliers.

- Consecutive-breach (>= global P95, consecutive by 1-minute, length >=2) findings:
  - JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed
    - earliest breach ts: 1616589000
    - segment length: 31 points (continuous across the incident window)
    - segment_max_value = global_P95 = 5.848308e+07 → severity = 1.0
  - JVM-Threads_7778_JVM_ThreadCount_Threads
    - breaches found (examples): earliest 1616590680 (length 2), severity ≈ 1.013
    - another segment at 1616590080 (length 2), severity = 1.0
  - OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc
    - multiple two-point segments (examples) including earliest 1616589000 and a segment at 1616589420
    - segment_max_value ≈ global_P95 (severity = 1.0) — indicates host memory % at or above historical P95 repeatedly.
  - A few system/process KPIs (zabbix process metric, NoCacheMemPerc) also show 2-point segments at P95 (severity = 1.0).

- Interpretation / likely root cause localization:
  - Evidence points to memory pressure on IG01:
    - Host memory used percentage repeatedly at or above P95 (~98% in global stats).
    - JVM non-heap usage (NoHeapMemoryUsed) sustained at its historical P95 for the entire incident window (not an isolated spike).
    - HeapMemoryUsed had a large value at 1616589420 (≈8.67e+08) but did not form a sustained consecutive exceedance in the incident window.
  - CPU and network metrics do not show comparable sustained exceedances in these results (P95 for CPU on relevant hosts is moderate).
  - Overall, the telemetry indicates memory-related issues (host memory saturation and sustained JVM non-heap usage) are the primary suspects.

- Recommended next steps (concise):
  - Collect JVM GC logs and non-heap/heap allocation traces for IG01 around 1616589000–1616590800.
  - Inspect OS memory and swap activity, processes consuming memory on IG01, and kernel OOM or swapping events.
  - Review recent deployments or configuration changes that could increase memory usage (e.g., larger caches, classloading, native allocations).
  - If confirmed, consider freeing memory, tuning JVM memory, increasing host memory, or redistributing workload.

If you want, I can:
- show the specific narrow-window metric samples for the top suspect KPIs (HeapMemoryUsed, NoHeapMemoryUsed, MEMUsedMemPerc) or
- extract GC/log entries around the earliest breach timestamp (1616589000) for IG01.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           kpi_name
0     IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage
1     IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed
2     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed
3     IG01                    JVM-Runtime_7778_JVM_JVM_Uptime
4     IG01           JVM-Threads_7778_JVM_ThreadCount_Threads
..     ...                                                ...
52    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps
53    IG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite
54    IG01    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil
55    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...
56    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvg...

[57 rows x 2 columns],    cmdb_id                                           kpi_name  count           p50           p95     max_value
0     IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  4.690916e+08  8.668898e+08  8.686124e+08
1     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.848308e+07  5.848308e+07  5.848308e+07
2     IG01                    JVM-Runtime_7778_JVM_JVM_Uptime     19  1.669004e+06  1.669791e+06  1.669844e+06
3     IG01    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec      9  3.537400e+04  6.481720e+04  6.781600e+04
4     IG01              OSLinux-OSLinux_MEMORY_MEMORY_UserMem     19  7.640000e+03  7.666200e+03  7.668000e+03
..     ...                                                ...    ...           ...           ...           ...
45    IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     13  3.000000e-01  3.000000e-01  3.000000e-01
46    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps      9  0.000000e+00  1.099800e-01  1.833000e-01
47    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...      8  0.000000e+00  6.500000e-02  1.000000e-01
48    IG01    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil      9  3.000000e-04  3.600000e-04  4.000000e-04
49    IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps     10  0.000000e+00  0.000000e+00  0.000000e+00

[50 rows x 6 columns],      timestamp cmdb_id                                           kpi_name         value
0   1616589420    IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.689980e+01
1   1616589420    IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.670007e+08
2   1616589480    IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  1.141281e+08
3   1616589420    IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.848308e+07
4   1616589480    IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.848308e+07
..         ...     ...                                                ...           ...
44  1616589420    IG01              OSLinux-OSLinux_MEMORY_MEMORY_UserMem  7.668000e+03
45  1616589420    IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00
46  1616589420    IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00
47  1616589420    IG01    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil  2.000000e-04
48  1616589420    IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01

[49 rows x 4 columns],                                              kpi_name  earliest_breach_ts  segment_length  segment_max_value    global_P95  severity
0            JVM-Threads_7778_JVM_ThreadCount_Threads          1616590680               2       7.600000e+01  7.500000e+01  1.013333
1         JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed          1616589000              31       5.848308e+07  5.848308e+07  1.000000
2            JVM-Threads_7778_JVM_ThreadCount_Threads          1616590080               2       7.500000e+01  7.500000e+01  1.000000
3        OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc          1616589000               2       9.800000e+01  9.800000e+01  1.000000
4        OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc          1616589180               2       9.800000e+01  9.800000e+01  1.000000
5        OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc          1616589420               2       9.800000e+01  9.800000e+01  1.000000
6        OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc          1616589600               2       9.800000e+01  9.800000e+01  1.000000
7        OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc          1616589780               2       9.800000e+01  9.800000e+01  1.000000
8        OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc          1616589720               2       2.830920e+01  2.830920e+01  1.000000
9   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...          1616590080               2       3.000000e-01  3.000000e-01  1.000000
10  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...          1616590620               2       3.000000e-01  3.000000e-01  1.000000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import re
import pytz

# Reuse trace_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Incident window
start_incident = 1616589000
end_incident   = 1616590800

# 1) Compute global thresholds from full trace series (do NOT use filtered window)
#    - duration -> global P95
#    - row_count -> global P5
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95),
         global_p5 =lambda x: x.quantile(0.05))
)

# 2) Filter trace rows to incident window and detect anomalies
trace_window_incident = trace_df[(trace_df['timestamp'] >= start_incident) & (trace_df['timestamp'] <= end_incident)].copy()

# Merge thresholds
trace_merged = trace_window_incident.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# Classify metric types
trace_merged['is_duration'] = trace_merged['trace_name'].str.contains('duration', case=False, na=False)
trace_merged['is_rowcount'] = trace_merged['trace_name'].str.contains('row_count', case=False, na=False)

# Duration anomalies: value >= global_p95
dur_anoms = trace_merged[trace_merged['is_duration'] & (trace_merged['value'] >= trace_merged['global_p95'])].copy()
# Row_count anomalies: value <= global_p5
rc_anoms  = trace_merged[trace_merged['is_rowcount'] & (trace_merged['value'] <= trace_merged['global_p5'])].copy()

# Helper to extract direction and target_component
def infer_direction_target(name):
    if 'trace.self' in name or name.startswith('trace.self'):
        return 'self', ''
    m_to = re.search(r'\bto_([A-Za-z0-9]+)', name)
    m_from = re.search(r'\bfrom_([A-Za-z0-9]+)', name)
    if m_to:
        return 'to', m_to.group(1)
    if m_from:
        return 'from', m_from.group(1)
    # fallback: try "to" and "from" with dot separator e.g., 'trace.to.Tomcat01' (unlikely here)
    m_to2 = re.search(r'\bto\.?([A-Za-z0-9]+)', name)
    m_from2 = re.search(r'\bfrom\.?([A-Za-z0-9]+)', name)
    if m_to2:
        return 'to', m_to2.group(1)
    if m_from2:
        return 'from', m_from2.group(1)
    return '', ''

# 3) Aggregate anomalies per (cmdb_id, trace_name)
agg_rows = []

# Aggregate duration anomalies (high)
if not dur_anoms.empty:
    g = dur_anoms.groupby(['cmdb_id','trace_name'], as_index=False)
    dur_agg = g.agg(
        anomaly_count_in_window = ('value','count'),
        earliest_anomaly_timestamp_in_window = ('timestamp','min'),
        window_max_value = ('value','max'),
        global_threshold_used = ('global_p95','first')
    )
    dur_agg['anomaly_type'] = 'high'
    # infer direction and target
    dur_agg[['direction','target_component']] = dur_agg['trace_name'].apply(lambda n: pd.Series(infer_direction_target(n)))
    dur_agg['severity'] = dur_agg.apply(lambda r: (np.inf if r['global_threshold_used']==0 else r['window_max_value'] / r['global_threshold_used']), axis=1)
    agg_rows.append(dur_agg)

# Aggregate row_count anomalies (low)
if not rc_anoms.empty:
    g2 = rc_anoms.groupby(['cmdb_id','trace_name'], as_index=False)
    rc_agg = g2.agg(
        anomaly_count_in_window = ('value','count'),
        earliest_anomaly_timestamp_in_window = ('timestamp','min'),
        window_min_value = ('value','min'),
        global_threshold_used = ('global_p5','first')
    )
    rc_agg = rc_agg.rename(columns={'window_min_value':'window_min_value'})
    rc_agg['anomaly_type'] = 'low'
    rc_agg[['direction','target_component']] = rc_agg['trace_name'].apply(lambda n: pd.Series(infer_direction_target(n)))
    # severity for low: global_p5 / window_min_value (handle zero)
    rc_agg['severity'] = rc_agg.apply(lambda r: (np.inf if r['window_min_value']==0 else r['global_threshold_used'] / r['window_min_value']), axis=1)
    # align column names to match duration aggregated frame (use window_min_value instead of window_max_value)
    # We'll keep window_min_value and later choose which to display based on anomaly_type
    agg_rows.append(rc_agg)

# Combine aggregates
if agg_rows:
    trace_anomaly_agg = pd.concat(agg_rows, ignore_index=True, sort=False)
else:
    trace_anomaly_agg = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count_in_window','earliest_anomaly_timestamp_in_window',
        'window_max_value','window_min_value','global_threshold_used','anomaly_type','direction','target_component','severity'
    ])

# Normalize columns: ensure window_extreme column exists for display (window_max_value for high, window_min_value for low)
def choose_window_extreme(row):
    if row['anomaly_type'] == 'high':
        return row.get('window_max_value', np.nan)
    else:
        return row.get('window_min_value', np.nan)

if not trace_anomaly_agg.empty:
    trace_anomaly_agg['window_extreme_value'] = trace_anomaly_agg.apply(choose_window_extreme, axis=1)

# 4) Final ordering and selection: sort by earliest_anomaly_timestamp asc, then severity desc; limit 20 rows
trace_anomaly_agg_final = trace_anomaly_agg.copy()
if not trace_anomaly_agg_final.empty:
    trace_anomaly_agg_final = trace_anomaly_agg_final.sort_values(['earliest_anomaly_timestamp_in_window','severity'], ascending=[True,False])
    trace_anomaly_agg_final = trace_anomaly_agg_final[[
        'cmdb_id',
        'trace_name',
        'direction',
        'target_component',
        'anomaly_type',
        'anomaly_count_in_window',
        'earliest_anomaly_timestamp_in_window',
        'window_extreme_value',
        'global_threshold_used',
        'severity'
    ]].head(20).reset_index(drop=True)
else:
    # empty frame with columns
    trace_anomaly_agg_final = pd.DataFrame(columns=[
        'cmdb_id','trace_name','direction','target_component','anomaly_type',
        'anomaly_count_in_window','earliest_anomaly_timestamp_in_window',
        'window_extreme_value','global_threshold_used','severity'
    ])

# Identify the most downstream faulty service:
# Among rows where target_component is non-empty (calls between services), pick the row with the earliest anomaly timestamp,
# break ties by highest severity. If none, return a message saying none found.
downstream_candidates = trace_anomaly_agg_final[trace_anomaly_agg_final['target_component'].astype(bool)].copy()
if not downstream_candidates.empty:
    min_ts = downstream_candidates['earliest_anomaly_timestamp_in_window'].min()
    earliest_rows = downstream_candidates[downstream_candidates['earliest_anomaly_timestamp_in_window'] == min_ts]
    chosen = earliest_rows.sort_values('severity', ascending=False).iloc[0]
    downstream_summary = (
        f"Most downstream faulty service candidate: target_component={chosen['target_component']} "
        f"(cmdb_id={chosen['cmdb_id']}, trace_name={chosen['trace_name']}, "
        f"earliest_ts={int(chosen['earliest_anomaly_timestamp_in_window'])}, severity={float(chosen['severity']):.3f})"
    )
else:
    downstream_summary = "No inter-service trace anomalies with a target_component found in the incident window."

# Store intermediate tables in kernel variables for follow-up
trace_thresholds = trace_thresholds
trace_window_incident = trace_window_incident
trace_merged = trace_merged
trace_anomaly_agg = trace_anomaly_agg
trace_anomaly_agg_final = trace_anomaly_agg_final

# Compact bounded output: aggregated anomalies (<=20 rows) and one-line downstream summary
trace_anomaly_agg_final, downstream_summary

```
Out[7]:
```
Summary of trace-based localization (incident window 1616589000 — 1616590800):

- Total: anomaly aggregation produced up to 20 compact trace anomaly rows (duration high or row_count low) across services in the incident window. Earliest anomalies start at timestamp 1616589000.

- Earliest/most prominent anomalies (examples):
  - Tomcat02: trace.from_root.duration_mean and .duration_p95 flagged high at 1616589000 (severity reported as infinite because the global threshold was zero — verify threshold data).
  - MG02 <-> dockerA2: trace.from_dockerA2.duration_mean and duration_p95 flagged high starting 1616589000 (anomaly_count=2). dockerA2 trace.to_MG02 entries mirror these (severity ≈ 2.14 for mean, ≈ 1.59 for p95).
  - MG01 <-> dockerA1: trace.from_dockerA1.row_count and dockerA1 trace.to_MG01.row_count flagged low (anomaly_count=4) at 1616589000 (severity 1.5).
  - dockerB1 self.duration_p95: strong local spike (anomaly_count=5) at 1616589060 with high severity ≈ 4.82.
  - MG02 <-> dockerB1: trace.from_dockerB1.duration_p95 (and dockerB1 -> MG02) flagged high at 1616589060 (severity ≈ 3.17).
  - Several trace.from_root.row_count and duration entries across Tomcat02/Tomcat03/dockerB2 also show anomalies at 1616589000.

- Most downstream faulty service candidate (one-line):
  - target_component = root (cmdb_id = Tomcat02, trace_name = trace.from_root.duration_mean), earliest_ts = 1616589000, severity reported = inf.
    - Note: "inf" arises because the global threshold used for that trace was zero — this needs verification (zero threshold may indicate absent historical data or a normalization artifact). If threshold is valid, the zero threshold should be addressed; otherwise recompute/inspect the baseline.

- Interpretation and guidance:
  - Inter-service latency anomalies cluster around a few call paths: dockerA2 ↔ MG02, dockerB1 ↔ MG02, dockerA1 ↔ MG01, and root → various Tomcat nodes. These suggest downstream latency/throughput problems impacting MG02 and root-call paths.
  - Where row_count is low (dockerA1 → MG01), traffic dropouts occurred at the same early timestamp (1616589000).
  - High-duration spikes on dockerB1 and MG02 paths at ~1616589060 indicate downstream slowdowns/latency amplification.
  - Because some anomalies use zero global thresholds (yielding infinite severity), validate global threshold calculations for those traces before relying on the numeric severity ranking.

- Recommended next steps:
  - Inspect traces around timestamps 1616589000 and 1616589060 for the highlighted call paths (dockerA2→MG02, dockerB1→MG02, dockerA1→MG01, root→Tomcat02).
  - Verify global threshold values (especially zero thresholds) and recompute if needed.
  - If trace spans include payloads or error details, extract example traces for the most severe target_component entries to find the root cause (service bottleneck, queueing, or downstream failures).

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name direction target_component anomaly_type  anomaly_count_in_window  earliest_anomaly_timestamp_in_window  window_extreme_value  global_threshold_used  severity
0   Tomcat02      trace.from_root.duration_mean      from             root         high                        1                            1616589000              0.000000               0.000000       inf
1   Tomcat02       trace.from_root.duration_p95      from             root         high                        1                            1616589000              0.000000               0.000000       inf
2       MG02  trace.from_dockerA2.duration_mean      from         dockerA2         high                        2                            1616589000              0.207500               0.096745  2.144814
3   dockerA2        trace.to_MG02.duration_mean        to             MG02         high                        2                            1616589000              0.207500               0.096745  2.144814
4       MG02   trace.from_dockerA2.duration_p95      from         dockerA2         high                        2                            1616589000              0.815000               0.512970  1.588787
5   dockerA2         trace.to_MG02.duration_p95        to             MG02         high                        2                            1616589000              0.815000               0.512970  1.588787
6       MG01      trace.from_dockerA1.row_count      from         dockerA1          low                        4                            1616589000              4.000000               6.000000  1.500000
7   dockerA1            trace.to_MG01.row_count        to             MG01          low                        4                            1616589000              4.000000               6.000000  1.500000
8   Tomcat03      trace.from_root.duration_mean      from             root         high                        1                            1616589000              0.343667               0.343667  1.000000
9   Tomcat03       trace.from_root.duration_p95      from             root         high                        1                            1616589000              0.908700               0.908700  1.000000
10  dockerB2      trace.from_root.duration_mean      from             root         high                        1                            1616589000              0.009000               0.009000  1.000000
11  dockerB2       trace.from_root.duration_p95      from             root         high                        1                            1616589000              0.009000               0.009000  1.000000
12  Tomcat02          trace.from_root.row_count      from             root          low                        1                            1616589000              1.000000               1.000000  1.000000
13  Tomcat03          trace.from_root.row_count      from             root          low                        1                            1616589000              3.000000               3.000000  1.000000
14  dockerB2          trace.from_root.row_count      from             root          low                        1                            1616589000              1.000000               1.000000  1.000000
15  dockerB1            trace.self.duration_p95      self                          high                        5                            1616589060              0.053000               0.011000  4.818182
16      MG02   trace.from_dockerB1.duration_p95      from         dockerB1         high                        2                            1616589060              3.685600               1.161275  3.173753
17  dockerB1         trace.to_MG02.duration_p95        to             MG02         high                        2                            1616589060              3.685600               1.161275  3.173753
18      MG01   trace.from_dockerA2.duration_p95      from         dockerA2         high                        2                            1616589060              3.143200               1.215850  2.585187
19  dockerA2         trace.to_MG01.duration_p95        to             MG01         high                        2                            1616589060              3.143200               1.215850  2.585187, 'Most downstream faulty service candidate: target_component=root (cmdb_id=Tomcat02, trace_name=trace.from_root.duration_mean, earliest_ts=1616589000, severity=inf)')```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import re
import pytz

# Reuse metric_df from kernel and timezone per rules
tz = pytz.timezone('Asia/Shanghai')

cmdb = 'IG01'
start_incident = 1616589000
end_incident   = 1616590800
target_ts = 1616589420

# 1) KPI substring list (exact substring matching as provided)
substrs = [
    'HeapMemoryUsed','HeapMemoryUsage','NoHeapMemoryUsed','JVM_JVM_CPULoad','JVM_JVM',
    'ThreadCount_Threads','gc','GC','log.gc_pause','CPUCpuUtil','OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc',
    'OSLinux-OSLinux_CPU_CPUCpuUtil','NETInErr','NETOutErr','NETKBTotalPerSec','FSCapacity',
    'DSK','DISK'
]
pattern = '(' + '|'.join(re.escape(s) for s in substrs) + ')'

# Find matching (cmdb_id, kpi_name) for IG01
metric_ig_resource_candidates = (
    metric_df[
        (metric_df['cmdb_id'] == cmdb) &
        metric_df['kpi_name'].str.contains(pattern, regex=True, na=False)
    ][['cmdb_id','kpi_name']]
    .drop_duplicates()
    .reset_index(drop=True)
    .head(200)  # limit to 200 rows
)

# 2) Global aggregates (full series) for these matching KPIs: count, P50, P95, max
candidate_kpis = metric_ig_resource_candidates['kpi_name'].unique().tolist()

metric_ig_resource_full = metric_df[
    (metric_df['cmdb_id'] == cmdb) &
    (metric_df['kpi_name'].isin(candidate_kpis))
].copy()

metric_ig_resource_global_stats = (
    metric_ig_resource_full
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(count='count',
         p50=lambda x: x.quantile(0.5),
         p95=lambda x: x.quantile(0.95),
         max_value='max')
    .sort_values('p95', ascending=False)
    .head(50)  # up to top 50 rows by P95 desc
    .reset_index(drop=True)
)

# Prepare global thresholds lookup (P95 and P5)
global_p95 = metric_ig_resource_global_stats.set_index('kpi_name')['p95'].to_dict()
# compute global P5 using full series grouped (rule 9: thresholds computed BEFORE filtering)
metric_ig_resource_p5 = (
    metric_ig_resource_full
    .groupby('kpi_name', as_index=False)['value']
    .agg(global_p5=lambda x: x.quantile(0.05))
).set_index('kpi_name')['global_p5'].to_dict()

# 3) Extract samples at target_ts and one minute before/after within incident window
sample_ts = [target_ts - 60, target_ts, target_ts + 60]
# ensure within incident bounds
sample_ts = [t for t in sample_ts if (t >= start_incident and t <= end_incident)]

metric_ig_resource_samples = metric_ig_resource_full[
    metric_ig_resource_full['timestamp'].isin(sample_ts)
][['timestamp','cmdb_id','kpi_name','value']].sort_values(['kpi_name','timestamp']).reset_index(drop=True)

# Limit samples to up to 3 per kpi and total 200
metric_ig_resource_samples = (
    metric_ig_resource_samples
    .groupby(['cmdb_id','kpi_name'], as_index=False, group_keys=False)
    .apply(lambda df: df.head(3))
    .reset_index(drop=True)
).head(200)

# 4) Detect consecutive anomaly segments in incident window (value >= global P95 -> high; 
#    or for traffic-like KPIs value <= global P5 -> low). Consecutive means timestamp diff == 60.
# Identify traffic-like KPIs by name patterns (KB, Tps, PerSec, row_count, NETKB, TotalPerSec)
traffic_pattern = re.compile(r'KB|Tps|PerSec|row_count|NETKB|TotalPerSec', re.IGNORECASE)

metric_ig_incident = metric_ig_resource_full[
    (metric_ig_resource_full['timestamp'] >= start_incident) &
    (metric_ig_resource_full['timestamp'] <= end_incident)
].copy().sort_values(['kpi_name','timestamp']).reset_index(drop=True)

flagged_segments = []
for kpi, dfk in metric_ig_incident.groupby('kpi_name'):
    gp95 = global_p95.get(kpi, np.nan)
    gp5  = metric_ig_resource_p5.get(kpi, np.nan)
    if np.isnan(gp95) and np.isnan(gp5):
        continue
    dfk = dfk.sort_values('timestamp').reset_index(drop=True)
    # determine anomaly type per-point: traffic-like -> low anomalies; else high anomalies
    is_traffic = bool(traffic_pattern.search(kpi))
    if is_traffic:
        dfk['is_anom'] = dfk['value'] <= gp5
    else:
        dfk['is_anom'] = dfk['value'] >= gp95
    anom = dfk[dfk['is_anom']].copy()
    if anom.empty:
        continue
    anom['ts_diff'] = anom['timestamp'].diff().fillna(99999).astype(int)
    anom['seg_id'] = (anom['ts_diff'] != 60).cumsum()
    segs = anom.groupby('seg_id').agg(
        segment_start_ts=('timestamp','min'),
        segment_end_ts=('timestamp','max'),
        segment_length=('timestamp','count'),
        segment_max_value=('value','max'),
        segment_min_value=('value','min')
    ).reset_index(drop=True)
    # keep only segments length >=2
    segs = segs[segs['segment_length'] >= 2]
    if segs.empty:
        continue
    # For reporting per KPI choose the segment with highest severity
    segs['global_p95'] = gp95
    segs['global_p5']  = gp5
    # compute severity: for high use max / gp95; for low use gp5 / min (handle zeros)
    if is_traffic:
        segs['severity'] = segs.apply(lambda r: (np.inf if r['segment_min_value'] == 0 else (r['global_p5'] / r['segment_min_value'])), axis=1)
    else:
        segs['severity'] = segs.apply(lambda r: (np.inf if r['global_p95'] == 0 else (r['segment_max_value'] / r['global_p95'])), axis=1)
    # pick segment with max severity
    best = segs.sort_values('severity', ascending=False).iloc[0]
    # earliest breach segment that starts at or before target_ts (if any)
    segs_sorted = segs.sort_values('segment_start_ts')
    starts_before_target = segs_sorted[segs_sorted['segment_start_ts'] <= target_ts]
    if not starts_before_target.empty:
        earliest_breach_ts = int(starts_before_target.iloc[0]['segment_start_ts'])
    else:
        earliest_breach_ts = np.nan
    flagged_segments.append({
        'kpi_name': kpi,
        'earliest_breach_ts_before_target': earliest_breach_ts,
        'segment_start_ts': int(best['segment_start_ts']),
        'segment_end_ts': int(best['segment_end_ts']),
        'segment_length': int(best['segment_length']),
        'segment_max_value': float(best['segment_max_value']),
        'global_P95': float(best['global_p95']) if not np.isnan(best['global_p95']) else np.nan,
        'global_P5': float(best['global_p5']) if not np.isnan(best['global_p5']) else np.nan,
        'severity': float(best['severity'])
    })

if flagged_segments:
    metric_ig_resource_flagged = pd.DataFrame(flagged_segments)
    metric_ig_resource_flagged = metric_ig_resource_flagged.sort_values('severity', ascending=False).head(20).reset_index(drop=True)
else:
    metric_ig_resource_flagged = pd.DataFrame(columns=[
        'kpi_name','earliest_breach_ts_before_target','segment_start_ts','segment_end_ts','segment_length',
        'segment_max_value','global_P95','global_P5','severity'
    ])

# Store variables in kernel as requested
metric_ig_resource_candidates = metric_ig_resource_candidates
metric_ig_resource_global_stats = metric_ig_resource_global_stats
metric_ig_resource_samples = metric_ig_resource_samples
metric_ig_resource_flagged = metric_ig_resource_flagged

# Compact bounded outputs (respect limits): candidates (<=200), global_stats (<=50), samples (<=200), flagged (<=20)
metric_ig_resource_candidates, metric_ig_resource_global_stats, metric_ig_resource_samples, metric_ig_resource_flagged

```
Out[8]:
```
```python
# Compact plain-English summary using existing kernel variables
candidates = metric_ig_resource_candidates
global_stats = metric_ig_resource_global_stats
samples = metric_ig_resource_samples
flagged = metric_ig_resource_flagged

num_kpis = len(candidates)
# pick top global by p95 examples (if available)
top_globals = []
for _, r in global_stats.head(3).iterrows():
    top_globals.append(f"{r['kpi_name']} (P95={r['p95']:.6g}, P50={r['p50']:.6g}, max={r['max_value']:.6g})")

# sample values at 1616589420: show a few key KPIs if present
def find_sample(k):
    df = samples
    s = df[df['kpi_name'].str.contains(k, na=False)]
    return s[['timestamp','kpi_name','value']].to_dict('records')[:3]

heap_used_sample = find_sample('HeapMemoryUsed')
noheap_sample = find_sample('NoHeapMemoryUsed')
heap_usage_sample = find_sample('HeapMemoryUsage')
memperc_sample = find_sample('MEMUsedMemPerc')

# flagged consecutive segments summary (top by severity)
flagged_lines = []
for _, r in flagged.head(10).iterrows():
    kb = r['kpi_name']
    eb = r['earliest_breach_ts_before_target']
    if pd.isna(eb):
        eb = 'none <= target_ts'
    flagged_lines.append(
        f"{kb} | start={r['segment_start_ts']} end={r['segment_end_ts']} len={r['segment_length']} "
        f"max={r['segment_max_value']:.6g} P95={r['global_P95']:.6g} severity={r['severity']:.6g}"
    )

summary_lines = [
    "IG01 resource-evidence report (focused around 1616589420):",
    "",
    f"1) Matching resource KPIs found: {num_kpis} unique (examples top-by-P95):",
]
summary_lines += [f"   - {g}" for g in top_globals] 

summary_lines += [
    "",
    "2) Narrow-window samples at/around 1616589420 (selected examples):",
]
if heap_used_sample:
    for s in heap_used_sample:
        summary_lines.append(f"   - {s['timestamp']} {s['kpi_name']} = {s['value']:.6g}")
else:
    summary_lines.append("   - No HeapMemoryUsed sample at target ts")

if noheap_sample:
    for s in noheap_sample:
        summary_lines.append(f"   - {s['timestamp']} {s['kpi_name']} = {s['value']:.6g}")
if heap_usage_sample:
    for s in heap_usage_sample:
        summary_lines.append(f"   - {s['timestamp']} {s['kpi_name']} = {s['value']:.6g}")
if memperc_sample:
    for s in memperc_sample:
        summary_lines.append(f"   - {s['timestamp']} {s['kpi_name']} = {s['value']:.6g}")

summary_lines += [
    "",
    "3) Flagged consecutive breach segments (length>=2) near the event (top by severity):"
]
if flagged_lines:
    for l in flagged_lines:
        summary_lines.append(f"   - {l}")
else:
    summary_lines.append("   - No flagged multi-point consecutive breaches detected.")

summary_lines += [
    "",
    "Conclusion:",
    "   - Evidence points to memory pressure on IG01 around the incident time:",
    "     • Host memory percent (MEMUsedMemPerc) is at P95 (~98%) with consecutive small segments including the event time.",
    "     • JVM non-heap (NoHeapMemoryUsed) is sustained at its historical P95 across the incident window (31 points) — sustained, not isolated.",
    "     • HeapMemoryUsed peaked at 1616589420 (~8.67e+08), near its global P95, indicating substantial heap usage at the event time.",
    "   - CPU, disk, and network KPIs show no comparable sustained exceedances in these results.",
    "",
    "Recommendation: collect IG01 JVM GC logs and OS memory/swap/process info around 1616589000–1616590800; investigate JVM non-heap and possible native allocations or memory leaks."
]

# Return the summary string
"\n".join(summary_lines)
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           kpi_name
0     IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage
1     IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed
2     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed
3     IG01                    JVM-Runtime_7778_JVM_JVM_Uptime
4     IG01           JVM-Threads_7778_JVM_ThreadCount_Threads
..     ...                                                ...
32    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead
33    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps
34    IG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite
35    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...
36    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvg...

[37 rows x 2 columns],    cmdb_id                                           kpi_name  count          p50          p95    max_value
0     IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  469091568.0  866889840.0  868612448.0
1     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31   58483080.0   58483080.0   58483080.0
2     IG01                    JVM-Runtime_7778_JVM_JVM_Uptime     19    1669004.0    1669790.9    1669844.0
3     IG01    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec      9      35374.0      64817.2      67816.0
4     IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...      9         84.0        113.2        122.0
..     ...                                                ...    ...          ...          ...          ...
32    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead      9          0.0          0.0          0.0
33    IG01            OSLinux-OSLinux_NETWORK_ens160_NETInErr     10          0.0          0.0          0.0
34    IG01         OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc     10          0.0          0.0          0.0
35    IG01           OSLinux-OSLinux_NETWORK_ens160_NETOutErr     10          0.0          0.0          0.0
36    IG01       OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc     10          0.0          0.0          0.0

[37 rows x 6 columns],      timestamp cmdb_id                                          kpi_name         value
0   1616589360    IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.178120e+01
1   1616589420    IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.689980e+01
2   1616589360    IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  7.528062e+08
3   1616589420    IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.670007e+08
4   1616589480    IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  1.141281e+08
..         ...     ...                                               ...           ...
41  1616589420    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  9.120000e+01
42  1616589420    IG01      OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01
43  1616589480    IG01      OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01
44  1616589360    IG01        OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc  0.000000e+00
45  1616589360    IG01      OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc  0.000000e+00

[46 rows x 4 columns],                                        kpi_name  earliest_breach_ts_before_target  segment_start_ts  segment_end_ts  segment_length  segment_max_value  global_P95   global_P5  severity
0      JVM-Threads_7778_JVM_ThreadCount_Threads                               NaN        1616590680      1616590740               2               76.0        75.0        68.0  1.013333
1   JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                      1.616589e+09        1616589000      1616590800              31         58483080.0  58483080.0  58483080.0  1.000000
2  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc                      1.616589e+09        1616589000      1616589060               2               98.0        98.0        97.0  1.000000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import re
import pytz

# Reuse existing kernel variable metric_df
tz = pytz.timezone('Asia/Shanghai')

cmdb = 'IG01'
start_incident = 1616589000
end_incident   = 1616590800
event_ts = 1616589420
sample_ts = [1616589360, 1616589420, 1616589480]

# 1) find matching KPI names for IG01 by exact substrings (case-sensitive as requested)
substrs = [
    'gc','gc_pause','HeapMemoryUsed','HeapMemoryUsage','NoHeapMemoryUsed',
    'JVM_JVM_CPULoad','ThreadCount_Threads','CPUCpuUtil',
    'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc',
    'swap','Swap','NETInErr','NETOutErr','NETKBTotalPerSec'
]
pattern = '(' + '|'.join(re.escape(s) for s in substrs) + ')'

metric_ig_resource_candidates = (
    metric_df[
        (metric_df['cmdb_id'] == cmdb) &
        metric_df['kpi_name'].str.contains(pattern, regex=True, na=False)
    ][['cmdb_id','kpi_name']]
    .drop_duplicates()
    .reset_index(drop=True)
    .head(200)
)

# 2) compute global aggregates (full series) for these KPIs: count, P50, P95, max
candidate_kpis = metric_ig_resource_candidates['kpi_name'].unique().tolist()

metric_ig_resource_full = metric_df[
    (metric_df['cmdb_id'] == cmdb) &
    (metric_df['kpi_name'].isin(candidate_kpis))
].copy()

metric_ig_resource_global_stats = (
    metric_ig_resource_full
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(count='count',
         p50=lambda x: x.quantile(0.5),
         p95=lambda x: x.quantile(0.95),
         max_value='max')
    .sort_values('p95', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# prepare global thresholds lookup (P95 and P5) computed from full series
global_p95_lookup = metric_ig_resource_global_stats.set_index('kpi_name')['p95'].to_dict()
global_p5_lookup = (
    metric_ig_resource_full
    .groupby('kpi_name', as_index=False)['value']
    .agg(global_p5=lambda x: x.quantile(0.05))
    .set_index('kpi_name')['global_p5']
    .to_dict()
)

# 3) extract up to 3 samples at specified timestamps (if present) for each candidate KPI (limit total rows 200)
metric_ig_resource_samples_event = (
    metric_ig_resource_full[metric_ig_resource_full['timestamp'].isin(sample_ts)]
    [['timestamp','cmdb_id','kpi_name','value']]
    .sort_values(['kpi_name','timestamp'])
    .reset_index(drop=True)
)

# limit to up to 3 per KPI (samples are at most 3 timestamps) and total cap 200
metric_ig_resource_samples_event = (
    metric_ig_resource_samples_event
    .groupby(['cmdb_id','kpi_name'], as_index=False, group_keys=False)
    .apply(lambda df: df.head(3))
    .reset_index(drop=True)
).head(200)

# 4) detect consecutive multi-point runs (length>=2) within incident window where:
#    - for traffic-like KPIs (row_count/KB/PerSec/Tps/NETKB/TotalPerSec) treat anomaly as value <= global_P5 (low)
#    - otherwise treat anomaly as value >= global_P95 (high)
traffic_pattern = re.compile(r'row_count|KB|PerSec|Tps|NETKB|TotalPerSec', re.IGNORECASE)

metric_ig_incident = metric_ig_resource_full[
    (metric_ig_resource_full['timestamp'] >= start_incident) &
    (metric_ig_resource_full['timestamp'] <= end_incident)
].copy().sort_values(['kpi_name','timestamp']).reset_index(drop=True)

flagged_records = []
for kpi, dfk in metric_ig_incident.groupby('kpi_name'):
    gp95 = global_p95_lookup.get(kpi, np.nan)
    gp5  = global_p5_lookup.get(kpi, np.nan)
    dfk = dfk.sort_values('timestamp').reset_index(drop=True)
    is_traffic = bool(traffic_pattern.search(kpi))
    if is_traffic:
        if np.isnan(gp5):
            continue
        dfk['is_anom'] = dfk['value'] <= gp5
    else:
        if np.isnan(gp95):
            continue
        dfk['is_anom'] = dfk['value'] >= gp95
    anom = dfk[dfk['is_anom']].copy()
    if anom.empty:
        # still compute breached_at_event for present event_ts
        breached_at_event = False
        if not dfk[dfk['timestamp'] == event_ts].empty:
            val = float(dfk[dfk['timestamp'] == event_ts]['value'].iloc[0])
            if is_traffic:
                breached_at_event = val <= gp5 if not np.isnan(gp5) else False
            else:
                breached_at_event = val >= gp95 if not np.isnan(gp95) else False
        continue
    anom['ts_diff'] = anom['timestamp'].diff().fillna(99999).astype(int)
    anom['seg_id'] = (anom['ts_diff'] != 60).cumsum()
    segs = anom.groupby('seg_id').agg(
        segment_start_ts=('timestamp','min'),
        segment_end_ts=('timestamp','max'),
        segment_length=('timestamp','count'),
        segment_max_value=('value','max'),
        segment_min_value=('value','min')
    ).reset_index(drop=True)
    segs = segs[segs['segment_length'] >= 2]
    if segs.empty:
        # no multi-point segments
        # still determine breached_at_event
        breached_at_event = False
        if not dfk[dfk['timestamp'] == event_ts].empty:
            val = float(dfk[dfk['timestamp'] == event_ts]['value'].iloc[0])
            if is_traffic:
                breached_at_event = val <= gp5 if not np.isnan(gp5) else False
            else:
                breached_at_event = val >= gp95 if not np.isnan(gp95) else False
        continue
    # For reporting pick earliest segment (and compute severity from the segment_max or min)
    segs = segs.sort_values('segment_start_ts').reset_index(drop=True)
    # find earliest segment that starts at or before event_ts if any; otherwise earliest overall
    segs_before_event = segs[segs['segment_start_ts'] <= event_ts]
    if not segs_before_event.empty:
        chosen = segs_before_event.iloc[0]
        earliest_breach_ts = int(chosen['segment_start_ts'])
    else:
        chosen = segs.iloc[0]
        earliest_breach_ts = int(chosen['segment_start_ts'])
    if is_traffic:
        severity = (np.inf if chosen['segment_min_value'] == 0 else (gp5 / chosen['segment_min_value']))
    else:
        severity = (np.inf if gp95 == 0 else (chosen['segment_max_value'] / gp95))
    # breached_at_event flag
    breached_at_event = False
    evrow = dfk[dfk['timestamp'] == event_ts]
    if not evrow.empty:
        val = float(evrow['value'].iloc[0])
        if is_traffic:
            breached_at_event = val <= gp5 if not np.isnan(gp5) else False
        else:
            breached_at_event = val >= gp95 if not np.isnan(gp95) else False
    flagged_records.append({
        'kpi_name': kpi,
        'breached_at_event': breached_at_event,
        'earliest_breach_ts': earliest_breach_ts,
        'segment_start_ts': int(chosen['segment_start_ts']),
        'segment_end_ts': int(chosen['segment_end_ts']),
        'segment_length': int(chosen['segment_length']),
        'segment_max_value': float(chosen['segment_max_value']),
        'segment_min_value': float(chosen['segment_min_value']),
        'global_P95': float(gp95) if not np.isnan(gp95) else np.nan,
        'global_P5': float(gp5) if not np.isnan(gp5) else np.nan,
        'severity': float(severity)
    })

# assemble flagged dataframe, sort by severity desc, limit 20
if flagged_records:
    metric_ig_resource_flagged = pd.DataFrame(flagged_records)
    metric_ig_resource_flagged = metric_ig_resource_flagged.sort_values('severity', ascending=False).head(20).reset_index(drop=True)
else:
    metric_ig_resource_flagged = pd.DataFrame(columns=[
        'kpi_name','breached_at_event','earliest_breach_ts','segment_start_ts','segment_end_ts',
        'segment_length','segment_max_value','segment_min_value','global_P95','global_P5','severity'
    ])

# Store variables as requested (they also serve as displayed outputs)
metric_ig_resource_candidates = metric_ig_resource_candidates
metric_ig_resource_global_stats = metric_ig_resource_global_stats
metric_ig_resource_samples_event = metric_ig_resource_samples_event
metric_ig_resource_flagged = metric_ig_resource_flagged

# Compact bounded display: four DataFrames (limited per requirements)
metric_ig_resource_candidates, metric_ig_resource_global_stats, metric_ig_resource_samples_event, metric_ig_resource_flagged

```
Out[9]:
```
```python
summary = (
    "IG01 memory/GC-focused snapshot around 1616589420 (compact):\n\n"
    "1) Matching KPIs (examples): HeapMemoryUsed, HeapMemoryUsage, NoHeapMemoryUsed, JVM CPU/Threads, "
    "OSLinux CPU, OS memory percent (MEMUsedMemPerc), swap, and network error/KB metrics.\n\n"
    "2) Key global aggregates (from full series):\n"
    "   - HeapMemoryUsed: P95 ≈ 8.6689e+08, max ≈ 8.6861e+08 (count 31).\n"
    "   - NoHeapMemoryUsed: P95 = 5.848308e+07, max = 5.848308e+07 (count 31).\n"
    "   - MEMUsedMemPerc (host memory %): P95 = 98.0% (count 19).\n"
    "   - ThreadCount_Threads: P95 = 75 (count 31). CPU P95 ≈ 25–26% (moderate).\n\n"
    "3) Three-point samples (≈4 min before, event, ≈4 min after) — selected values at 1616589420:\n"
    "   - JVM HeapMemoryUsed @1616589420 = 8.670007e+08 (near global P95).\n"
    "   - JVM NoHeapMemoryUsed @1616589420 = 5.848308e+07 (equal to global P95).\n"
    "   - MEMUsedMemPerc @1616589420 = 98.0% (at P95 baseline).\n"
    "   - JVM thread count and CPU show modest values (threads ~74, CPU ~25%). Swap metrics show zeros.\n\n"
    "4) Flagged consecutive multi-point breaches (length>=2) relevant to the event:\n"
    "   - JVM-Memory ... NoHeapMemoryUsed: breached at event (True); segment start 1616589000 — end 1616590800; length 31; "
    "segment_max = global_P95 = 5.848308e+07; severity = 1.0 (sustained at P95 across the incident).\n"
    "   - OSLinux MEMORY MEMUsedMemPerc: breached at event (True); example 2-point segments including 1616589420; "
    "segment_max = 98.0%; severity = 1.0 (host memory at historical 95th percentile).\n"
    "   - JVM ThreadCount has short 2-point segments but not necessarily at the event timestamp.\n\n"
    "Interpretation and next steps (concise):\n"
    " - Telemetry points to memory pressure on IG01 at the suspected time: host memory % is at P95 and JVM non-heap is sustained at P95.\n"
    " - HeapMemoryUsed peaked at the event timestamp (≈8.67e+08) but the strongest sustained signal is NoHeapMemoryUsed and OS memory saturation.\n"
    " - CPU, disk, and network metrics do not show comparable sustained exceedances; swap usage appears zero in sampled metrics.\n"
    " - Action: collect JVM GC logs, native/non-heap allocation traces and OS memory/swap/process details around 1616589000–1616590800; investigate non-heap allocations, GC behavior, and any recent deploys that could cause increased non-heap/native memory usage."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                         kpi_name
0     IG01       JVM-Memory_7778_JVM_Memory_HeapMemoryUsage
1     IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed
2     IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed
3     IG01         JVM-Threads_7778_JVM_ThreadCount_Threads
4     IG01                       OSLinux-CPU_CPU_CPUCpuUtil
5     IG01     OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc
6     IG01       OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc
7     IG01     OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc
8     IG01  OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent
9     IG01        JVM-Operating System_7778_JVM_JVM_CPULoad
10    IG01          OSLinux-OSLinux_NETWORK_ens160_NETInErr
11    IG01  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec
12    IG01         OSLinux-OSLinux_NETWORK_ens160_NETOutErr,    cmdb_id                                         kpi_name  count           p50           p95     max_value
0     IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  4.690916e+08  8.668898e+08  8.686124e+08
1     IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.848308e+07  5.848308e+07  5.848308e+07
2     IG01  OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec      9  3.537400e+04  6.481720e+04  6.781600e+04
3     IG01     OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc     19  9.800000e+01  9.800000e+01  9.800000e+01
4     IG01         JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.100000e+01  7.500000e+01  7.600000e+01
5     IG01       JVM-Memory_7778_JVM_Memory_HeapMemoryUsage     21  2.299320e+01  4.236320e+01  4.248630e+01
6     IG01                       OSLinux-CPU_CPU_CPUCpuUtil     21  2.577230e+01  2.604450e+01  2.624800e+01
7     IG01        JVM-Operating System_7778_JVM_JVM_CPULoad     14  2.778500e-01  3.165850e-01  3.329000e-01
8     IG01          OSLinux-OSLinux_NETWORK_ens160_NETInErr     10  0.000000e+00  0.000000e+00  0.000000e+00
9     IG01       OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc     10  0.000000e+00  0.000000e+00  0.000000e+00
10    IG01         OSLinux-OSLinux_NETWORK_ens160_NETOutErr     10  0.000000e+00  0.000000e+00  0.000000e+00
11    IG01     OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc     10  0.000000e+00  0.000000e+00  0.000000e+00
12    IG01  OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent     10  0.000000e+00  0.000000e+00  0.000000e+00,      timestamp cmdb_id                                         kpi_name         value
0   1616589360    IG01       JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.178120e+01
1   1616589420    IG01       JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.689980e+01
2   1616589360    IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  7.528062e+08
3   1616589420    IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.670007e+08
4   1616589480    IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  1.141281e+08
5   1616589360    IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.848308e+07
6   1616589420    IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.848308e+07
7   1616589480    IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.848308e+07
8   1616589420    IG01        JVM-Operating System_7778_JVM_JVM_CPULoad  2.497000e-01
9   1616589360    IG01         JVM-Threads_7778_JVM_ThreadCount_Threads  7.300000e+01
10  1616589420    IG01         JVM-Threads_7778_JVM_ThreadCount_Threads  7.400000e+01
11  1616589480    IG01         JVM-Threads_7778_JVM_ThreadCount_Threads  7.400000e+01
12  1616589360    IG01                       OSLinux-CPU_CPU_CPUCpuUtil  2.574030e+01
13  1616589420    IG01                       OSLinux-CPU_CPU_CPUCpuUtil  2.572250e+01
14  1616589420    IG01     OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01
15  1616589480    IG01     OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01
16  1616589360    IG01       OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc  0.000000e+00
17  1616589360    IG01     OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc  0.000000e+00
18  1616589360    IG01  OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent  0.000000e+00,                                        kpi_name  breached_at_event  earliest_breach_ts  segment_start_ts  segment_end_ts  segment_length  segment_max_value  segment_min_value  global_P95   global_P5  severity
0   JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed               True          1616589000        1616589000      1616590800              31         58483080.0         58483080.0  58483080.0  58483080.0       1.0
1      JVM-Threads_7778_JVM_ThreadCount_Threads              False          1616590080        1616590080      1616590140               2               75.0               75.0        75.0        68.0       1.0
2  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc               True          1616589000        1616589000      1616589060               2               98.0               98.0        98.0        97.0       1.0)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing trace_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global thresholds from full trace series (before filtering)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95),
         global_p5 =lambda x: x.quantile(0.05))
)

# 2) Filter trace rows to the narrow window and selected components
components = ['IG01','Tomcat01','Tomcat04','MG01','MG02','apache01','apache02']
start_ts = 1616589400
end_ts   = 1616589480

trace_narrow_df = trace_df[
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts) &
    (trace_df['cmdb_id'].isin(components))
].copy()

# Merge thresholds into the narrow window rows
trace_narrow_merged = trace_narrow_df.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# 3) Determine anomaly status per row
trace_narrow_merged['is_duration'] = trace_narrow_merged['trace_name'].str.contains('duration', case=False, na=False)
trace_narrow_merged['is_rowcount'] = trace_narrow_merged['trace_name'].str.contains('row_count', case=False, na=False)

def determine_anom_and_sev(row):
    val = row['value']
    if row['is_duration']:
        gp = row['global_p95']
        anom = False if pd.isna(gp) else (val >= gp)
        if pd.isna(gp):
            sev = np.nan
        else:
            sev = np.inf if gp == 0 else (val / gp)
        threshold = gp
        atype = 'high'
    elif row['is_rowcount']:
        gp5 = row['global_p5']
        anom = False if pd.isna(gp5) else (val <= gp5)
        if pd.isna(gp5):
            sev = np.nan
        else:
            sev = np.inf if val == 0 else (gp5 / val)
        threshold = gp5
        atype = 'low'
    else:
        # not classified => no anomaly
        anom = False
        sev = np.nan
        threshold = np.nan
        atype = ''
    return pd.Series([anom, atype, threshold, sev])

trace_narrow_merged[['is_anomaly','anomaly_type','global_threshold_used','severity']] = trace_narrow_merged.apply(determine_anom_and_sev, axis=1)

# 4) Filter to anomaly rows only and prepare compact output columns
trace_narrow_anomalies = trace_narrow_merged[trace_narrow_merged['is_anomaly']].copy()

# Select requested columns and sort by timestamp asc then severity desc
trace_narrow_anomalies = trace_narrow_anomalies[
    ['timestamp','cmdb_id','trace_name','value','global_threshold_used','anomaly_type','severity']
].sort_values(['timestamp','severity'], ascending=[True,False]).reset_index(drop=True)

# Limit to 50 rows for display
trace_narrow_anomalies_top = trace_narrow_anomalies.head(50)

# Build one-line earliest-anomaly summary
if not trace_narrow_anomalies.empty:
    earliest_ts = int(trace_narrow_anomalies['timestamp'].min())
    candidates = trace_narrow_anomalies[trace_narrow_anomalies['timestamp'] == earliest_ts]
    # pick highest severity among earliest
    chosen = candidates.sort_values('severity', ascending=False).iloc[0]
    earliest_anomaly_summary = (
        f"earliest_anomaly: timestamp={int(chosen['timestamp'])}, cmdb_id={chosen['cmdb_id']}, "
        f"trace_name={chosen['trace_name']}, anomaly_type={chosen['anomaly_type']}, severity={float(chosen['severity']):.6g}"
    )
else:
    earliest_anomaly_summary = "No anomalies detected in the narrow window for selected components."

# Store intermediate variables in kernel (per instructions)
trace_narrow_df = trace_narrow_df
trace_narrow_anomalies = trace_narrow_anomalies

# Compact outputs: anomaly table (<=50 rows) and one-line earliest summary
trace_narrow_anomalies_top, earliest_anomaly_summary

```
Out[10]:
```
```python
# Compose a concise plain-English summary using existing kernel variables
# (trace_narrow_anomalies_top, earliest_anomaly_summary are available from previous step)

rows_count = trace_narrow_anomalies.shape[0] if 'trace_narrow_anomalies' in globals() else 0

# prepare top anomalies examples (timestamp | cmdb_id | trace_name | type | severity)
top_examples = []
for _, r in trace_narrow_anomalies_head := (trace_narrow_anomalies.head(5) if 'trace_narrow_anomalies' in globals() else pd.DataFrame()).iterrows():
    top_examples.append(f"{int(r[1]['timestamp'])} | {r[1]['cmdb_id']} | {r[1]['trace_name']} | {r[1]['anomaly_type']} | sev={float(r[1]['severity']):.3g}")

summary = (
    "Trace-based localization (narrow window 1616589400–1616589480) — concise findings:\n\n"
    f"1) Total anomaly rows (selected components) in window: {rows_count}.\n\n"
    f"2) Earliest anomaly (by timestamp) in this result:\n   - {earliest_anomaly_summary}\n\n"
    "3) Other notable anomalies at the same event time (examples):\n"
    "   - MG02 trace.to_dockerA1.row_count was low at 1616589420 (severity ~1.082), indicating reduced traffic on that call path.\n"
    "   - IG01 trace.self.duration_mean was high at 1616589420 (severity ~1.085), indicating increased internal latency on IG01 itself.\n"
    "   - IG01 trace.to_Tomcat01.row_count and Tomcat01 trace.from_IG01.row_count were low at 1616589420 (severity ~1.082), showing decreased traffic between IG01 and Tomcat01.\n\n"
    "4) Interpretation:\n"
    "   - The earliest/highest-severity anomaly points to MG01 (trace.from_dockerB2.duration_p95 at 1616589420), suggesting downstream latency on the dockerB2→MG01 path.\n"
    "   - Simultaneously, IG01 shows self-duration increase and reduced outgoing row_count to Tomcat01, consistent with a downstream latency or service slowdown that reduces throughput.\n"
    "   - Multiple services show low row_count (traffic drop) and some high durations clustered at the same timestamps, indicating a cross-service latency/throughput impact rather than a pure client-side spike.\n\n"
    "5) Recommended immediate next steps:\n"
    "   - Inspect traces/spans for MG01 and dockerB2 around 1616589420 to see full call stacks and latency propagation.\n"
    "   - Fetch IG01 detailed traces and logs at 1616589420 to correlate the self-duration spike with downstream calls (Tomcat01) and error logs.\n"
    "   - Validate global thresholds that were zero (causing infinite severity) and re-evaluate if needed; focus on MG01/dockerB2 and IG01→Tomcat01 paths.\n\n"
    "Conclusion: The most downstream faulty candidate based on earliest anomaly is MG01 (incoming from dockerB2) at 1616589420; IG01 also exhibits latency and reduced outgoing traffic to Tomcat01 at the same time — investigate MG01/dockerB2 and IG01 traces first."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp   cmdb_id                        trace_name      value  global_threshold_used anomaly_type  severity
0   1616589420      MG01  trace.from_dockerB2.duration_p95   0.624200               0.473850         high  1.317295
1   1616589420      MG02       trace.to_dockerA1.row_count  35.000000              38.500000          low  1.100000
2   1616589420      IG01          trace.self.duration_mean   4.314705               3.975726         high  1.085262
3   1616589420      IG01       trace.to_Tomcat01.row_count  67.000000              72.500000          low  1.082090
4   1616589420  Tomcat01         trace.from_IG01.row_count  67.000000              72.500000          low  1.082090
..         ...       ...                               ...        ...                    ...          ...       ...
21  1616589480      MG01       trace.to_dockerB2.row_count  24.000000              24.500000          low  1.020833
22  1616589480      IG01           trace.self.duration_p95  15.584100              15.579500         high  1.000295
23  1616589480      MG01     trace.from_Tomcat04.row_count  77.000000              77.000000          low  1.000000
24  1616589480      MG01       trace.to_dockerA2.row_count  34.000000              34.000000          low  1.000000
25  1616589480  Tomcat04           trace.to_MG01.row_count  77.000000              77.000000          low  1.000000

[26 rows x 7 columns], 'earliest_anomaly: timestamp=1616589420, cmdb_id=MG01, trace_name=trace.from_dockerB2.duration_p95, anomaly_type=high, severity=1.31729')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```